In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import matplotlib.ticker as mticker
!pip install scikit-learn --break-system-packages

# Analysis Part 1

In [ ]:
df = pd.read_csv("Dataset 1 UVA.csv", sep=";")

In [ ]:
df.head()


In [ ]:
df['geboortedatum'] = df['geboortedatum'].replace(
    ['0', '0-0-0', '0000-00-00'],
    pd.NA
)

In [ ]:
df['geboortedatum'].isna().mean() * 100

In [ ]:
df.head()

In [ ]:
df = df.rename(columns={
    'geslacht': 'gender',
    'geboortedatum': 'birth_date',
    'postcode': 'postcode',
    'woonplaats': 'city',
    'interesses': 'interests'
})

Gender analysis

In [ ]:
df['gender'].value_counts(dropna=False)

In [ ]:
df['gender'] = df['gender'].str.strip().str.lower()

df['gender'] = df['gender'].replace({
    'v': 'female',
    'mevrouw': 'female',
    'm': 'male',
    'de heer': 'male',
    'o': 'other'
})

In [ ]:
df['gender'].value_counts(dropna=False)

In [ ]:
df['gender'].value_counts(normalize=True)

In [ ]:
from datetime import datetime

df['birth_date'] = pd.to_datetime(df['birth_date'], errors='coerce')
today = pd.to_datetime('today')

df['age'] = (today - df['birth_date']).dt.days // 365

In [ ]:
import matplotlib.pyplot as plt

df['age'].dropna().hist(bins=30)
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.title('Age Distribution')
plt.show()

In [ ]:
df_interests = df.copy()

df_interests['interests'] = df_interests['interests'].str.split(',')
df_interests = df_interests.explode('interests')

In [ ]:
df_interests['interests'] = (
    df_interests['interests']
    .str.strip()
    .str.lower()
)

Unique interests

In [ ]:
df_interests['interests'].nunique()

In [ ]:
df_interests['interests'].unique()

In [ ]:
df_interests['interests'].value_counts()

In [ ]:
df['postcode'] = (
    df['postcode']
    .astype(str)
    .str.upper()
    .str.replace(' ', '', regex=False)
    .str.extract(r'(\d{4}[A-Z]{0,2})')
)

In [ ]:
df['postcode'].sample(50)

In [ ]:
df['city'] = (
    df['city']
    .astype(str)
    .str.strip()
    .str.lower()
    .replace('null', pd.NA)
)

In [ ]:
df['city'] = df['city'].replace('nan', pd.NA)

In [ ]:
df['interests'] = df['interests'].str.split(',')

In [ ]:
type(df['interests'].iloc[0])

In [ ]:
df['interests'] = df['interests'].apply(
    lambda x: [i.strip().lower() for i in x] if isinstance(x, list) else []
)

In [ ]:
df['n_interests'] = df['interests'].apply(len)

In [ ]:
df.head()

# Campaign Analysis 

### Data prep

In [ ]:
# inspect (in)valid values in opens and clicks
import re

def classify_value(x):
    if pd.isna(x) or str(x).strip() == '':
        return 'missing'
    
    x = str(x).strip()

    # valid id
    if re.search(r'\b1:\d{3,4}\b', x):
        return 'id_format'
    
    return 'non_id_format'

In [ ]:
df['open_type'] = df['opens'].apply(classify_value)
df['click_type'] = df['clicks'].apply(classify_value)

print(df['open_type'].value_counts())
print(df['click_type'].value_counts())

In [ ]:
print(df['mailings'].iloc[0])

# function to extract id from mailings, opens, clicks
def extract_ids(text):
    if pd.isna(text):
        return []
    items = text.split(',')
    ids = []

    for item in items:
        item = item.strip()
        if ':' in item:
            ids.append(int(item.split(':')[1]))
    
    return ids

In [ ]:
# parse mailings, open, clicks
df['mailing_ids'] = df['mailings'].apply(extract_ids)
df['open_ids'] = df.apply(
    lambda row: extract_ids(row['opens']) if row['open_type'] == 'id_format' else [],
    axis = 1)
df['click_ids'] = df.apply(
    lambda row: extract_ids(row['clicks']) if row['click_type'] == 'id_format' else [],
    axis = 1)

In [ ]:
df[['opens', 'open_type', 'open_ids', 'clicks', 'click_type', 'click_ids']].head(26)

In [ ]:
# rename column id
df = df.rename(columns = {'id': 'user_id'})

In [ ]:
# explode dataset
df_exploded = df.explode('mailing_ids')
df_exploded = df_exploded.rename(columns = {'mailing_ids': 'mailing_id'})

df_exploded[['user_id', 'mailing_id']].head()

In [ ]:
import numpy as np

In [ ]:
# create three state open column: 1 if mailing_id was opened, 0 if not opened, NaN if invalid
def check_open(row):
    if row['open_type'] == 'missing':
        return 0
    
    if row['open_type'] != 'id_format':
        return np.nan
    
    if row['mailing_id'] in row['open_ids']:
        return 1
    else:
        return 0
    
df_exploded['open'] = df_exploded.apply(check_open, axis=1)

df_exploded[['user_id', 'mailing_id', 'open', 'open_ids']].head()

In [ ]:
# create three state click column
def check_click(row):
    if row['click_type'] == 'missing':
        return 0
    
    if row['click_type'] != 'id_format':
        return np.nan
    
    if row['mailing_id'] in row['click_ids']:
        return 1
    else:
        return 0
    
df_exploded['click'] = df_exploded.apply(check_click, axis=1)

df_exploded[['user_id', 'mailing_id', 'open', 'click', 'open_ids', 'click_ids']].head()

In [ ]:
df_exploded[['user_id', 'mailing_id', 'open', 'click']].head()

In [ ]:
print(df_exploded['open'].value_counts(dropna = False))
print(df_exploded['click'].value_counts(dropna = False))

In [ ]:
# load dataset 2
df2 = pd.read_excel('DATSET 2 UVA lijst mailings.xlsx')

In [ ]:
# rename columns
df2 = df2.rename(columns = {
    'ID': 'mailing_id',
    'Mailing': 'mailing_info',
    'Subjectline': 'subject_line',
    'Preheader': 'preheader'
})

In [ ]:
# merge dataset 1 and dataset 2
df_final = df_exploded.merge(
    df2,
    on = 'mailing_id',
    how = 'left'
)

In [ ]:
print(df_final[['open', 'click']].isna().sum())
print(df_final[['open', 'click']].dtypes)

In [ ]:
df_final[['user_id', 'mailing_id', 'open', 'click', 'subject_line']].head()

In [ ]:
# drop unnecessary columns
df_final = df_final.drop(columns = [
    'mailings',
    'opens',
    'clicks',
    'open_ids',
    'click_ids'
])

In [ ]:
print(df_final.columns)
df_final.head()

### Overall engagement rates

In [ ]:
# subjecline effectiveness (known cases only)
print('Open rate:', df_final['open'].mean())

# content effectiveness (known cases only)
print('Click rate:', df_final['click'].mean())

In [ ]:
print('Open unknown %:', df_final['open'].isna().mean())
print('Click unknown %:', df_final['click'].isna().mean())

In [ ]:
# subjectline

result = df_final.groupby(['subject_line', 'preheader']).agg(
    open_mean = ('open', 'mean'),
    click_mean = ('click', 'mean'),
    count_valid = ('open', 'count'),    #excludes NaN
    count_total = ('open', 'size')      #includes everything
)

# top subjectline being sent
result.sort_values('count_valid', ascending = False)[
    ['count_valid', 'count_total', 'open_mean', 'click_mean']
].head(10)

#### Top subject lines by open rate

In [ ]:
# top subjectline by open rate
result.sort_values('open_mean', ascending = False)[
    ['open_mean', 'click_mean', 'count_valid', 'count_total']
].head(10)

#### Top subject lines by click rate

In [ ]:
# top subjectline by click rate
result.sort_values('click_mean', ascending = False)[
    ['click_mean', 'open_mean', 'count_valid', 'count_total']
].head(10)

### Subject line vs Engagement segmentation

In [ ]:
print(result['open_mean'].describe())
print(result['click_mean'].describe())

In [ ]:
# threshold for engagement rates
open_threshold = result['open_mean'].mean()
click_threshold = result['click_mean'].mean()

In [ ]:
# classify subject lines into groups

result['open_group'] = result['open_mean'].apply(
    lambda x: 'high' if x > open_threshold else 'low')

result['click_group'] = result['click_mean'].apply(
    lambda x: 'high' if x > click_threshold else 'low')

In [ ]:
# size check
result.groupby(['open_group', 'click_group']).size()

In [ ]:
# extract groups
high_open_high_click = result[
    (result['open_group'] == 'high') &
    (result['click_group'] == 'high')]

high_open_low_click = result[
    (result['open_group'] == 'high') &
    (result['click_group'] == 'low')]

low_open_high_click = result[
    (result['open_group'] == 'low') &
    (result['click_group'] == 'high')]

low_open_low_click = result[
    (result['open_group'] == 'low') &
    (result['click_group'] == 'low')]


#### High open + High click group

In [ ]:
high_open_high_click[['open_mean', 'click_mean', 'count_valid', 'count_total']].sort_values('click_mean', ascending = False).head(10)

In [ ]:
high_open_low_click[['open_mean', 'click_mean', 'count_valid', 'count_total']].sort_values('open_mean', ascending = False).head(10)

#### High open + Low click group

In [ ]:
low_open_high_click[['open_mean', 'click_mean', 'count_valid', 'count_total']].sort_values('click_mean', ascending = False).head(10)

#### Low open + High click group

##### Insights 
Low open, high click group
- Subject lines focus on specific, niche topics (security, finance)
- May not attract broad attention (low open rate)
- Generate stronger engagement among a small, relevant group of audience (high click rate)

In [ ]:
low_open_low_click[['open_mean', 'click_mean', 'count_valid', 'count_total']].sort_values('open_mean', ascending = True).head(10)

#### Low open + Low click group

# User Engagement Analysis

Explore how user characteristics (gender, age, and interests) relate to engagement metrics

### Overview

#### Data preparation

In [ ]:
# open and click rates per user
df_user = df_final.groupby('user_id').agg(
    open_rate = ('open', 'mean'),
    click_rate = ('click', 'mean'),
    total_emails = ('open', 'count')
).reset_index()

# add user info back
df_user = df_user.merge(
    df_final[['user_id', 'gender', 'age', 'interests']].drop_duplicates(subset = 'user_id'),
    on = 'user_id',
    how = 'left'
)

In [ ]:
print(df_user['open_rate'].describe())
print(df_user['click_rate'].describe())

In [ ]:
df_user.isna().sum()

In [ ]:
print(df_final.groupby('user_id')['open'].apply(
    lambda x: set(x.dropna().unique())
).value_counts())

print(df_final.groupby('user_id')['click'].apply(
    lambda x: set(x.dropna().unique())
).value_counts())

In [ ]:
# drop users with unsuable engagement data
df_user_clean = df_user.dropna(subset = ['open_rate', 'click_rate'])
df_user_clean.shape

In [ ]:
print(df_user[df_user['open_rate'].isna() & df_user['click_rate'].notna()].shape)
print(df_user[df_user['click_rate'].isna() & df_user['open_rate'].notna()].shape)

In [ ]:
print(df_user_clean['open_rate'].describe())
print(df_user_clean['click_rate'].describe())

In [ ]:
df_user_clean[['open_rate', 'click_rate']].hist(bins=50)

In [ ]:
# click rate quantile check
df_user_clean['click_rate'].quantile([
    0.5, 0.75, 0.9, 0.95, 0.99, 0.999])

In [ ]:
# click rate quantile check for users who DO click
df_user_clean[df_user_clean['click_rate'] > 0]['click_rate'].quantile([
    0.5, 0.75, 0.9, 0.95, 0.99])

In [ ]:
print('% of users who ever click:', (df_user_clean['click_rate'] > 0).mean())

#### Classify users into engagment segmentation

In [ ]:
# threshold for high and low open
open_threshold = df_user_clean['open_rate'].median()        #median because open rate is reasonably spread, but not symmetric

In [ ]:
df_user_clean = df_user.dropna(subset=['open_rate', 'click_rate']).copy()

In [ ]:
# create user segment

# open segment
df_user_clean['open_segment'] = np.where(
    df_user_clean['open_rate'] >= open_threshold,
    'high_open',
    'low_open')

# click segment
def classify_click(x):
    if x == 0:
        return 'no_click'
    else:
        return 'click'
    
df_user_clean['click_segment'] = df_user_clean['click_rate'].apply(classify_click)

In [ ]:
# segment size
df_user_clean.groupby(['open_segment', 'click_segment']).size()

### Gender Analysis

In [ ]:
df_user_clean.groupby(
    ['open_segment', 'click_segment'])['gender'].value_counts(normalize = True)

In [ ]:
# bar chart

# data prep
gender_distribution = df_user_clean.groupby(
    ['open_segment','click_segment','gender']).size().unstack(fill_value = 0)

# convert to proportions
gender_distribution = gender_distribution.div(gender_distribution.sum(axis = 1), axis = 0)

# plot
gender_distribution.plot(kind = 'bar', stacked = True, width = 0.6)

# rename x labels
plt.xticks(
    ticks=range(len(gender_distribution.index)),
    labels=[
        'High Open\nClick',
        'High Open\nNo Click',
        'Low Open\nClick',
        'Low Open\nNo Click'],
    rotation=0)

plt.title('Gender Distribution across User Segments')
plt.ylabel('Proportion')
plt.xlabel('User Segments')
plt.legend(title='Gender', bbox_to_anchor = (1.05, 1), loc = 'upper left')
plt.tight_layout()
plt.show()

### Age Analysis

In [ ]:
df_user_clean.columns

In [ ]:
# remove unrealistic age
df_user_clean['age_clean'] = df_user_clean['age'].where(
    (df_user_clean['age'] >= 18) &
    (df_user_clean['age'] <= 90))

In [ ]:
# create age group
df_user_clean['age_group'] = pd.cut(
    df_user_clean['age_clean'],
    bins = [18, 35, 50, 65, 90],
    labels = ['18-35', '35-50', '50-65', '65+'])

# count users per age group
df_user_clean['age_group'].value_counts(normalize = True)

#### Engagement rate for age groups

In [ ]:
# engagement rate for age groups
df_user_clean.groupby('age_group', observed = True)[['open_rate', 'click_rate']].mean()

In [ ]:
# bar chart

# data prep
age_rates = df_user_clean.groupby('age_group', observed=True)[['open_rate', 'click_rate']].mean()

#plot
age_rates.plot(kind='bar')

plt.title('Engagement Rates by Age Group')
plt.ylabel('Average Rate')
plt.xlabel('Age Group')
plt.xticks(rotation=0)
plt.legend(['Open Rate', 'Click Rate'])
plt.tight_layout()
plt.show()

#### Age groups vs Engagement segmentation

In [ ]:
# age segment size
age_segment = pd.crosstab(
    df_user_clean['age_group'],
    [df_user_clean['open_segment'], df_user_clean['click_segment']],
    normalize = 'index')

age_segment

In [ ]:
df_user_clean['combined_segment'] = (
    df_user_clean['open_segment'] + ' & ' + df_user_clean['click_segment'])

age_by_segment = pd.crosstab(
    df_user_clean['combined_segment'],
    df_user_clean['age_group'],
    normalize = 'index')

# plot
age_by_segment.plot(kind = 'bar', stacked = True, width = 0.6)

# rename x labels
plt.xticks(
    ticks=range(len(age_by_segment.index)),
    labels=[
        'High Open\nClick',
        'High Open\nNo Click',
        'Low Open\nClick',
        'Low Open\nNo Click'],
    rotation=0)

plt.title('Age Distribution across User Segments')
plt.ylabel('Proportion')
plt.xlabel('User Segments')
plt.legend(title='Age Group', bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()

### Interest Analysis

#### Explore top interests among users

In [ ]:
# explode data with interests
df_interests = df_user_clean.copy()
df_interests = df_interests.explode('interests')

In [ ]:
# top user interests
top_interests = df_interests['interests'].value_counts().head(10)
print(top_interests)

In [ ]:
# bar chart

# english translation
interest_map = {
    'bladen': 'Magazines',
    'acties': 'Promotions',
    'kranten': 'Newspaper',
    'winactie': 'Giveaways',
    'cadeau': 'Gifts',
    'kortingen': 'Discounts',
    'energie': 'Energy',
    'goededoelen': 'Charity',
    'onderzoek': 'Research',
    'auto': 'Automotive'}

top_interests_en = top_interests.rename(index = interest_map)

#plot
top_interests_en.plot(kind='barh')
plt.title('Top 10 User Interests')
plt.xlabel('Count')
plt.ylabel('Interest')
plt.gca().invert_yaxis()
plt.show()

##### Insights
- Promotional interests dominate user preferences (promotions, giveaways, gifts, discounts)
- Users show strong interests in informational content (magazines, newspaper, research)
- Commercial topics are also significant (energy and automotive)

#### Explore interests with high engagment rates

In [ ]:
# aggregate engagement per interest
interest_engagement = df_interests.groupby('interests').agg(
    open_mean=('open_rate', 'mean'),
    click_mean=('click_rate', 'mean'),
    count=('user_id', 'count'))

interest_engagement.head()

In [ ]:
interest_counts = df_interests['interests'].value_counts()
interest_counts.describe()

In [ ]:
# remove small counts

#threshold for counts
count_threshold = threshold = df_interests['interests'].value_counts().quantile(0.25)
interest_filtered = interest_engagement[
    interest_engagement['count'] >= count_threshold]

In [ ]:
# sort interests by open rates
interest_filtered.sort_values('open_mean', ascending=False).head(10)

In [ ]:
# sort interests by click rates
interest_filtered.sort_values('click_mean', ascending=False).head(10)

Some overlap between popular and interests with high engagement, but the relationship is not strong or consistent

# Analysis Part 2

## Gender x Campaign Cross Analysis

In [ ]:
# get the campaign group per mailing
df_final_with_groups = df_final.merge(
    result[['open_group', 'click_group']].reset_index(), 
    on=['subject_line', 'preheader'],
    how='left')

# gender distribution within each campaign group
gender_campaign_group = df_final_with_groups.groupby(
    ['open_group', 'click_group', 'gender']
).agg(
    open_rate=('open', 'mean'),
    click_rate=('click', 'mean'),
    count=('open', 'count')
).reset_index()

print(gender_campaign_group)

In [ ]:
# Pivot: open rate per gender per campaign group
pivot_gender_campaign = df_final_with_groups.groupby(
    ['open_group', 'click_group', 'gender']
)['open'].mean().unstack('gender')

print("Open rate by campaign group Ã— gender:")
print(pivot_gender_campaign)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Open rate
pivot_open = df_final_with_groups.groupby(
    ['open_group', 'click_group', 'gender']
)['open'].mean().unstack('gender')

pivot_open.plot(kind='bar', ax=axes[0], color=['#4681E0', '#DE8551', '#7ED893'])
axes[0].set_title('Open Rate by Campaign Group — Gender')
axes[0].set_ylabel('Open Rate')
axes[0].set_xticklabels([
    'High Open\nHigh Click', 'High Open\nLow Click',
    'Low Open\nHigh Click', 'Low Open\nLow Click'],
    rotation=0)

# Click rate
pivot_click = df_final_with_groups.groupby(
    ['open_group', 'click_group', 'gender']
)['click'].mean().unstack('gender')

pivot_click.plot(kind='bar', ax=axes[1], color=['#4681E0', '#DE8551', '#7ED893'])
axes[1].set_title('Click Rate by Campaign Group — Gender')
axes[1].set_ylabel('Click Rate')
axes[1].set_xticklabels([
    'High Open\nHigh Click', 'High Open\nLow Click',
    'Low Open\nHigh Click', 'Low Open\nLow Click'],
    rotation=0)

plt.tight_layout()
plt.show()

## Age Group x Campaign Cross Analysis

In [ ]:
# Merge age group into df_final
df_final_enriched = df_final_with_groups.merge(
    df_user_clean[['user_id', 'age_group', 'age_clean']].drop_duplicates(),
    on='user_id',
    how='left'
)

# Open and click rate by age group — campaign group
age_campaign = df_final_enriched.groupby(
    ['age_group', 'open_group', 'click_group'],
    observed=True
).agg(
    open_rate=('open', 'mean'),
    click_rate=('click', 'mean'),
    count=('open', 'count')
).reset_index()

print(age_campaign)

In [ ]:
# Heatmap: open rate age group — campaign group
pivot_age_open = df_final_enriched.groupby(
    ['age_group', 'open_group'],
    observed=True
)['open'].mean().unstack('open_group')

pivot_age_click = df_final_enriched.groupby(
    ['age_group', 'click_group'],
    observed=True
)['click'].mean().unstack('click_group')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(
    pivot_age_open, annot=True, fmt='.2f',
    cmap='YlOrRd', ax=axes[0])
axes[0].set_title('Open Rate: Age Group — Campaign Open Group')

sns.heatmap(
    pivot_age_click, annot=True, fmt='.2f',
    cmap='YlOrRd', ax=axes[1])
axes[1].set_title('Click Rate: Age Group — Campaign Click Group')

plt.tight_layout()
plt.show()

## Interest x Campaign Cross Analysis

In [ ]:
top_n_interests = df_interests['interests'].value_counts().head(15).index.tolist()

# Explode interests in enriched dataset
df_exploded_interests = df_final_enriched.copy()
df_exploded_interests['interests'] = df_exploded_interests['interests'].apply(
    lambda x: x if isinstance(x, list) else []
)
df_exploded_interests = df_exploded_interests.explode('interests')
df_exploded_interests = df_exploded_interests[
    df_exploded_interests['interests'].isin(top_n_interests)
]

# Open and click rate by interest — campaign group
interest_campaign = df_exploded_interests.groupby(
    ['interests', 'open_group', 'click_group']
).agg(
    open_rate=('open', 'mean'),
    click_rate=('click', 'mean'),
    count=('open', 'count')
).reset_index()

In [ ]:
# Heatmap: interest — campaign group open rate
pivot_interest_open = df_exploded_interests.groupby(
    ['interests', 'open_group']
)['open'].mean().unstack('open_group')

pivot_interest_click = df_exploded_interests.groupby(
    ['interests', 'click_group']
)['click'].mean().unstack('click_group')

fig, axes = plt.subplots(1, 2, figsize=(14, 8))

sns.heatmap(
    pivot_interest_open, annot=True, fmt='.2f',
    cmap='YlOrRd', ax=axes[0])
axes[0].set_title('Open Rate: Interest — Campaign Open Group')

sns.heatmap(
    pivot_interest_click, annot=True, fmt='.2f',
    cmap='YlOrRd', ax=axes[1])
axes[1].set_title('Click Rate: Interest — Campaign Click Group')

plt.tight_layout()
plt.show()

In [ ]:
# Which interests are overrepresented in high open/click campaigns?
high_perf = df_exploded_interests[
    (df_exploded_interests['open_group'] == 'high') &
    (df_exploded_interests['click_group'] == 'high')
]
low_perf = df_exploded_interests[
    (df_exploded_interests['open_group'] == 'low') &
    (df_exploded_interests['click_group'] == 'low')
]

interest_lift = pd.DataFrame({
    'high_perf_rate': high_perf['interests'].value_counts(normalize=True),
    'low_perf_rate': low_perf['interests'].value_counts(normalize=True)
}).dropna()

interest_lift['lift'] = (
    interest_lift['high_perf_rate'] / interest_lift['low_perf_rate']
)

print("\nInterest lift  high vs low performing campaigns:")
print(interest_lift.sort_values('lift', ascending=False))

## Interest x Gender - Engagement Analysis

In [ ]:
df_interest_user = df_user_clean.copy()
df_interest_user = df_interest_user.explode('interests')
df_interest_user = df_interest_user[
    df_interest_user['interests'].isin(top_n_interests)
]

# Open rate by interest Ã— gender
pivot_interest_gender_open = df_interest_user.groupby(
    ['interests', 'gender']
)['open_rate'].mean().unstack('gender')

pivot_interest_gender_click = df_interest_user.groupby(
    ['interests', 'gender']
)['click_rate'].mean().unstack('gender')

fig, axes = plt.subplots(1, 2, figsize=(14, 8))

sns.heatmap(
    pivot_interest_gender_open, annot=True, fmt='.2f',
    cmap='Blues', ax=axes[0])
axes[0].set_title('Open Rate: Interest — Gender')

sns.heatmap(
    pivot_interest_gender_click, annot=True, fmt='.2f',
    cmap='Oranges', ax=axes[1])
axes[1].set_title('Click Rate: Interest — Gender')

plt.tight_layout()
plt.show()

## Interest x Age Group - Engagement Analysis

In [ ]:
pivot_interest_age_open = df_interest_user.groupby(
    ['interests', 'age_group'],
    observed=True
)['open_rate'].mean().unstack('age_group')

pivot_interest_age_click = df_interest_user.groupby(
    ['interests', 'age_group'],
    observed=True
)['click_rate'].mean().unstack('age_group')

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

sns.heatmap(
    pivot_interest_age_open, annot=True, fmt='.2f',
    cmap='Blues', ax=axes[0])
axes[0].set_title('Open Rate: Interest — Age Group')

sns.heatmap(
    pivot_interest_age_click, annot=True, fmt='.2f',
    cmap='Oranges', ax=axes[1])
axes[1].set_title('Click Rate: Interest — Age Group')

plt.tight_layout()
plt.show()

## Email Frequency Fatigue - Demographics Analysis

In [ ]:
# Bin total emails received into frequency groups
df_user_clean['frequency_group'] = pd.cut(
    df_user_clean['total_emails'],
    bins=[0, 5, 10, 20, 50, 999],
    labels=['1-5', '6-10', '11-20', '21-50', '50+']
)

# Fatigue curve overall
fatigue_overall = df_user_clean.groupby(
    'frequency_group',
    observed=True
)[['open_rate', 'click_rate']].mean()

print("Fatigue effect overall:")
print(fatigue_overall)



In [ ]:
# Fatigue — gender
fatigue_gender = df_user_clean.groupby(
    ['frequency_group', 'gender'],
    observed=True
)[['open_rate', 'click_rate']].mean()

pivot_fatigue_gender = fatigue_gender['open_rate'].unstack('gender')

# Fatigue — age group
fatigue_age = df_user_clean.groupby(
    ['frequency_group', 'age_group'],
    observed=True
)[['open_rate', 'click_rate']].mean()

pivot_fatigue_age = fatigue_age['open_rate'].unstack('age_group')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pivot_fatigue_gender.plot(ax=axes[0], marker='o')
axes[0].set_title('Email Fatigue by Gender')
axes[0].set_ylabel('Average Open Rate')
axes[0].set_xlabel('Emails Received')

pivot_fatigue_age.plot(ax=axes[1], marker='o')
axes[1].set_title('Email Fatigue by Age Group')
axes[1].set_ylabel('Average Open Rate')
axes[1].set_xlabel('Emails Received')

plt.tight_layout()
plt.show()

## Campaign Topic Analysis

### Data preparation: classification into campaigns 

In [ ]:
# create campaign topic table
df_campaigns = df2.copy()
campaign_topics = df_campaigns[['mailing_id', 'mailing_info', 'subject_line', 'preheader']].copy()

# drop duplicates
campaign_topics = campaign_topics.drop_duplicates(subset = ['mailing_id'])

In [ ]:
campaign_topics.head()

In [ ]:
campaign_topics.shape

In [ ]:
# add classification column
campaign_topics['main_topic'] = ''

In [ ]:
# combine mailing info, subject line, preheader
campaign_topics['combined_info'] = (
    campaign_topics['mailing_info'].fillna('') + '' + 
    campaign_topics['subject_line'].fillna('') + '' + 
    campaign_topics['preheader'].fillna(''))

In [ ]:
campaign_topics[
    campaign_topics['combined_info'].str.contains(
        'Mazda|Audi|Hyundai|Toyota|mobiliteit', 
        case = False, 
        na = False,
        regex = True)
][['mailing_id', 'combined_info']]

In [ ]:
# classify automotive/mobility topics
campaign_topics.loc[
    campaign_topics['combined_info'].str.contains(
        'Mazda|Audi|Hyundai|Toyota|mobiliteit', 
        case = False, 
        na = False,
        regex = True),
        'main_topic'
] = 'Automotive & Mobility'

In [ ]:
campaign_topics[['mailing_id', 'main_topic']].head(22)

In [ ]:
campaign_topics[
    campaign_topics['combined_info'].str.contains(
        'magazine|krant|film|media|De Telegraaf|Libelle|Donald|economie|VARAgids|politiek|landleven|wetenschap', 
        case = False, 
        na = False,
        regex = True)
][['mailing_id', 'combined_info']]

In [ ]:
# classify media/publishing topics
campaign_topics.loc[
    campaign_topics['combined_info'].str.contains(
        'magazine|krant|film|media|De Telegraaf|Libelle|Donald|economie|VARAgids|politiek|landleven|wetenschap', 
        case = False, 
        na = False,
        regex = True),
        'main_topic'
] = 'Media & Publishing'

In [ ]:
campaign_topics[['mailing_id', 'main_topic']].head(22)

In [ ]:
campaign_topics[
    campaign_topics['combined_info'].str.contains(
        'inkomen|pensioen|Roadway|capital|invest|beleggen|AOV|Tikkie|autoverzekering', 
        case = False, 
        na = False,
        regex = True)
][['mailing_id', 'combined_info']]

In [ ]:
# classify finance/investment topics
campaign_topics.loc[
    campaign_topics['combined_info'].str.contains(
        'inkomen|pensioen|Roadway|capital|invest|beleggen|AOV|Tikkie|autoverzekering', 
        case = False, 
        na = False,
        regex = True),
        'main_topic'
] = 'Finance & Investment'

In [ ]:
campaign_topics[['mailing_id', 'main_topic']].head(22)

In [ ]:
campaign_topics[
    campaign_topics['combined_info'].str.contains(
        'Engie|zonne|powerbank|airco', 
        case = False, 
        na = False,
        regex = True)
][['mailing_id', 'combined_info']]

In [ ]:
# classify energy/sustainability topics
campaign_topics.loc[
    campaign_topics['combined_info'].str.contains(
        'Engie|zonne|powerbank|airco', 
        case = False, 
        na = False,
        regex = True),
        'main_topic'
] = 'Energy & Sustainability'

In [ ]:
campaign_topics[['mailing_id', 'main_topic']].head(22)

In [ ]:
campaign_topics['main_topic'].value_counts(dropna = False)

In [ ]:
campaign_topics[
    campaign_topics['combined_info'].str.contains(
        'goede doelen|hulp|Lepra|Oranje Fond|burendag', 
        case = False, 
        na = False,
        regex = True)
][['mailing_id', 'combined_info']]

In [ ]:
# classify charity/social impact topics
campaign_topics.loc[
    campaign_topics['combined_info'].str.contains(
        'goede doelen|hulp|Lepra|Oranje Fond|burendag', 
        case = False, 
        na = False,
        regex = True),
        'main_topic'
] = 'Charity & Social Impact'

In [ ]:
campaign_topics[['mailing_id', 'main_topic']].head(22)

In [ ]:
campaign_topics[
    campaign_topics['combined_info'].str.contains(
        'MKbasics|RANDSTAD|Regus|zaken|ziekteverzuim|groeien', 
        case = False, 
        na = False,
        regex = True)
][['mailing_id', 'combined_info']]

In [ ]:
# classify business topics
campaign_topics.loc[
    campaign_topics['combined_info'].str.contains(
        'MKbasics|RANDSTAD|Regus|zaken|ziekteverzuim|groeien', 
        case = False, 
        na = False,
        regex = True),
        'main_topic'
] = 'Business'

In [ ]:
campaign_topics[['mailing_id', 'main_topic']].head(22)

In [ ]:
campaign_topics[
    campaign_topics['combined_info'].str.contains(
        'Nespresso|Uitgekookt', 
        case = False, 
        na = False,
        regex = True)
][['mailing_id', 'combined_info']]

In [ ]:
# classify food/beverages topics
campaign_topics.loc[
    campaign_topics['combined_info'].str.contains(
        'Nespresso|Uitgekookt', 
        case = False, 
        na = False,
        regex = True),
        'main_topic'
] = 'Food & Beverages'

In [ ]:
campaign_topics[['mailing_id', 'main_topic']].head(22)

In [ ]:
campaign_topics[
    campaign_topics['combined_info'].str.contains(
        'Odido', 
        case = False, 
        na = False,
        regex = True)
][['mailing_id', 'combined_info']]

In [ ]:
# classify telecom/technology topics
campaign_topics.loc[
    campaign_topics['combined_info'].str.contains(
        'Odido', 
        case = False, 
        na = False,
        regex = True),
        'main_topic'
] = 'Telecom & Technology'

In [ ]:
campaign_topics[['mailing_id', 'main_topic']].head(22)

In [ ]:
campaign_topics[
    campaign_topics['combined_info'].str.contains(
        'Belife|mentaal|Pearle', 
        case = False, 
        na = False,
        regex = True)
][['mailing_id', 'combined_info']]

In [ ]:
# classify health/wellbeing topics
campaign_topics.loc[
    campaign_topics['combined_info'].str.contains(
        'Belife|mentaal|Pearle', 
        case = False, 
        na = False,
        regex = True),
        'main_topic'
] = 'Health & Wellbeing'

In [ ]:
campaign_topics[['mailing_id', 'main_topic']].head(22)

In [ ]:
campaign_topics[
    campaign_topics['combined_info'].str.contains(
        'loterij|Stempunt|prijzenrad|lifepoints|e-bike|geluk', 
        case = False, 
        na = False,
        regex = True)
][['mailing_id', 'combined_info']]

In [ ]:
# classify lottery/games topics
campaign_topics.loc[
    campaign_topics['combined_info'].str.contains(
        'loterij|Stempunt|prijzenrad|lifepoints|e-bike|geluk', 
        case = False, 
        na = False,
        regex = True),
        'main_topic'
] = 'Lottery & Games'

In [ ]:
campaign_topics[['mailing_id', 'main_topic']].head(22)

In [ ]:
campaign_topics['main_topic'].value_counts(dropna = False)

In [ ]:
campaign_topics[
    campaign_topics['combined_info'].str.contains(
        'bespaartips|deals|Welke deal past bij jou deze week?', 
        case = False, 
        na = False,
        regex = True)
][['mailing_id', 'combined_info']]

In [ ]:
# classify retail/promotion topics
campaign_topics.loc[
    campaign_topics['combined_info'].str.contains(
        'bespaartips|deals|Welke deal past bij jou deze week?', 
        case = False, 
        na = False,
        regex = True),
        'main_topic'
] = 'Retail & Promotion'

In [ ]:
campaign_topics[['mailing_id', 'main_topic']].head(22)

In [ ]:
# classify unknown topics
campaign_topics.loc[
    campaign_topics['main_topic'].str.strip() == '',
    'main_topic'
] = 'Unknown'

In [ ]:
campaign_topics[['mailing_id', 'main_topic']].head(22)

In [ ]:
campaign_topics['main_topic'].value_counts(dropna = False)

### Campaign topics distribution

In [ ]:
# topics distribution
topic_counts = campaign_topics['main_topic'].value_counts(dropna = False)
print(topic_counts)

In [ ]:
# plot

topic_counts = campaign_topics['main_topic'].value_counts().sort_values()

topic_counts.plot(kind='barh')
plt.title('Distribution of Campaign Topics')
plt.xlabel('Number of Campaigns')
plt.ylabel('Campaign Topic')
plt.tight_layout()
plt.show()

### Campaign topic - User distribution

In [ ]:
campaign_topics.columns

In [ ]:
df_final.columns

In [ ]:
# merge 2 datasets
df_final_topic = df_final.merge(
    campaign_topics[['mailing_id','main_topic']],
    on = 'mailing_id',
    how = 'left'
)

df_final_topic[['mailing_id', 'main_topic', 'open', 'click']].head()

In [ ]:
df_final_topic['main_topic'].isna().sum()

In [ ]:
df_final_topic[df_final_topic['main_topic'].isna()][
    ['mailing_id', 'mailing_info', 'subject_line', 'preheader']
].drop_duplicates()

In [ ]:
# relabel missing mailing id topis
df_final_topic['main_topic'] = df_final_topic['main_topic'].fillna('Missing')

In [ ]:
# topic exposure stats
topic_size = (
    df_final_topic.groupby('main_topic').agg({
        'mailing_id': 'nunique',     # number of campaigns
        'user_id': 'count'           # total user exposures
    })
    .rename(columns = {
        'mailing_id': '#campaigns',
        'user_id': '#user_exposures'
    })
)

In [ ]:
# average campaign size
topic_size['avg_campaign_size'] = (topic_size['#user_exposures'] / topic_size['#campaigns'])

# sort by average campaign size
topic_size.sort_values('avg_campaign_size', ascending = False)

In [ ]:
# scatter plot
plt.figure(figsize=(10, 6))

plt.scatter(
    topic_size['#user_exposures'],
    topic_size['#campaigns'])

# labels
for topic, row in topic_size.iterrows():
    plt.text(
        row['#user_exposures'],
        row['#campaigns'],
        topic,
        fontsize=9)
    
# log scale for x axis
plt.xscale('log')

plt.title('User Exposures vs Campaign Frequency by Topic')
plt.xlabel('Number of User Exposures (log scale)')
plt.ylabel('Number of Campaigns')

plt.grid(True)

plt.show()

## Topic x Engagement Analysis

#### Unweighted topic x engagement analysis (not account for number of user exposures)

In [ ]:
print(df_final_topic['open'].value_counts(dropna = False))
print(df_final_topic['click'].value_counts(dropna = False))

In [ ]:
# campaign - open rate and click rate
campaign_performance = (
    df_final_topic.groupby(['mailing_id', 'main_topic']).agg({
        'open': 'mean',
        'click': 'mean'
    }).reset_index())

campaign_performance.head()

In [ ]:
# topic - open and click rates
topic_performance = (
    campaign_performance.groupby('main_topic').agg({
        'open': ['mean', 'median'],
        'click': ['mean', 'median'],
        'mailing_id': 'count'
    }))

topic_performance

In [ ]:
# topics with highest average open rate
topic_performance.sort_values(('open', 'mean'), ascending = False)

In [ ]:
# topics with highest average click rate
topic_performance[['click', 'open', 'mailing_id']].sort_values(('click', 'mean'), ascending = False)

#### Weighted topic x engagement analysis (account for number of user exposures)

In [ ]:
# topic - open and click rates (weighted)
weighted_topic_performance = (
    df_final_topic.groupby('main_topic').agg({
        'open': 'mean',
        'click': 'mean',
        'user_id': 'count'})
        .rename(columns = {
            'user_id': "#user_exposures",
            'open': 'open_weighted',
            'click': 'click_weighted'}))

weighted_topic_performance

In [ ]:
# topics with highest average open rate (weighted)
weighted_topic_performance.sort_values('open_weighted', ascending = False)

In [ ]:
# topics with highest average click rate (weighted)
weighted_topic_performance.sort_values('click_weighted', ascending = False)

## Topic x User Analysis (Engagement segmentation Analysis)

In [ ]:
# merge topic dataset and user dataset
df_final_topic = df_final_topic.merge(
    df_user_clean[['user_id', 'open_segment', 'click_segment']],
    on = 'user_id',
    how = 'left')

df_final_topic[['user_id', 'main_topic', 'open_segment', 'click_segment']].head()

In [ ]:
# topic distribution by opens
topic_open_segment = pd.crosstab(
    df_final_topic['main_topic'],
    df_final_topic['open_segment'],
    normalize = 'columns')

# difference between high and low open column
topic_open_segment['open_difference'] = (topic_open_segment['high_open'] - topic_open_segment['low_open'])

# sort by difference
topic_open_segment.reindex(topic_open_segment['open_difference'].abs().sort_values(ascending = False).index)

In [ ]:
# topic distribution by clicks
topic_click_segment = pd.crosstab(
    df_final_topic['main_topic'],
    df_final_topic['click_segment'],
    normalize = 'columns')

# difference between high and low open column
topic_click_segment['click_difference'] = (topic_click_segment['click'] - topic_click_segment['no_click'])

# sort by strongest difference
topic_click_segment.reindex(topic_click_segment['click_difference'].abs().sort_values(ascending = False).index)

## Topic x Gender Analysis

### Topic distribution by gender

In [ ]:
# topic distribution by gender
topic_gender = pd.crosstab(
    df_final_topic['main_topic'],
    df_final_topic['gender'],
    normalize = 'columns')

In [ ]:
# male vs female difference
topic_gender['distribution_difference'] = (topic_gender['female'] - topic_gender['male'])

# sort by strongest difference
topic_gender = topic_gender.reindex(topic_gender['distribution_difference'].abs().sort_values(ascending = False).index)

topic_gender

### Topic x Gender Engagement Analysis

In [ ]:
df_final_topic['gender'].value_counts(dropna = False)

In [ ]:
# engagement rates by topic and gender
topic_gender_engagement = (df_final_topic.groupby(['main_topic', 'gender']).agg({
    'open': 'mean',
    'click': 'mean'}).reset_index())

topic_gender_engagement.head(6)

In [ ]:
# pivot open rate
topic_gender_open = topic_gender_engagement.pivot(
    index = 'main_topic',
    columns = 'gender',
    values = 'open')

topic_gender_open

In [ ]:
# female vs male difference
topic_gender_open['open_difference'] = (topic_gender_open['female'] - topic_gender_open['male'])

# sort by strongest difference
topic_gender_open = topic_gender_open.reindex(topic_gender_open['open_difference'].abs().sort_values(ascending = False).index)

topic_gender_open

In [ ]:
# pivot click rate
topic_gender_click = topic_gender_engagement.pivot(
    index = 'main_topic',
    columns = 'gender',
    values = 'click')

topic_gender_click

In [ ]:
# female vs male difference
topic_gender_click['click_difference'] = (topic_gender_click['female'] - topic_gender_click['male'])

# sort by strongest difference
topic_gender_click = topic_gender_click.reindex(topic_gender_click['click_difference'].abs().sort_values(ascending = False).index)

topic_gender_click

## Topic x Age Analysis

In [ ]:
# keep realistic ages only (18-90)
df_age = df_final_topic[
    (df_final_topic['age'] >= 18) &
    (df_final_topic['age'] <= 90)].copy()

# create age groups
df_age['age_group'] = pd.cut(
    df_age['age'],
    bins = [18, 35, 50, 65, 90],
    labels = ['18-35', '35-50', '50-65', '65+'])

In [ ]:
# remove missing age
df_age = df_age.dropna(subset = ['age'])

In [ ]:
df_age['age_group'].value_counts(normalize = True)

### Topic distribution by age group

In [ ]:
# topic distribution by age group
topic_age = pd.crosstab(
    df_age['main_topic'],
    df_age['age_group'],
    normalize = 'columns')

topic_age

### Topic x Age Engagement Analysis

In [ ]:
# engagement rates by topic, age groups
topic_age_engagement = (
    df_age.groupby(['main_topic', 'age_group']).agg({
        'open': 'mean',
        'click': 'mean'}).reset_index())

topic_age_engagement.head(8)

In [ ]:
# pivot open rate
topic_age_open = topic_age_engagement.pivot(
    index = 'main_topic',
    columns = 'age_group',
    values = 'open')

In [ ]:
# youngest vs oldest open rate difference
topic_age_open['open_difference'] = (topic_age_open['18-35'] - topic_age_open['65+'])

# sort by strongest difference
topic_age_open = topic_age_open.reindex(topic_age_open['open_difference'].abs().sort_values(ascending = False).index)

topic_age_open

In [ ]:
# pivot click rate
topic_age_click = topic_age_engagement.pivot(
    index = 'main_topic',
    columns = 'age_group',
    values = 'click')

In [ ]:
# youngest vs oldest click rate difference
topic_age_click['click_difference'] = (topic_age_click['18-35'] - topic_age_click['65+'])

# sort by strongest difference
topic_age_click = topic_age_click.reindex(topic_age_click['click_difference'].abs().sort_values(ascending = False).index)

topic_age_click

In [ ]:
print(df_age.groupby('age_group')['click'].sum())
print('\n', df_age.groupby('age_group')['click'].mean())

## Interest relevance analysis

In [ ]:
df_final_topic.columns

In [ ]:
df_final_topic.head()

In [ ]:
# interests dictionary
interest_topic_map = {
    # automotive
    'auto': 'Automotive & Mobility',
    'cx80': 'Automotive & Mobility',
    'auto inruilen': 'Automotive & Mobility',
    'mazda': 'Automotive & Mobility',
    'elektrisch': 'Automotive & Mobility',
    'audi': 'Automotive & Mobility',
    'mazda6e': 'Automotive & Mobility',
    'dacia': 'Automotive & Mobility',
    'hybrid': 'Automotive & Mobility',
    'mercedes': 'Automotive & Mobility',
    'toyota': 'Automotive & Mobility',
    'lease': 'Automotive & Mobility',
    'cx60': 'Automotive & Mobility',
    'cx5': 'Automotive & Mobility',
    'mx30': 'Automotive & Mobility',
    'cx30': 'Automotive & Mobility',
    'hyundai': 'Automotive & Mobility',
    'bedrijfswagen': 'Automotive & Mobility',
    'porsche': 'Automotive & Mobility',


    # business
    'mkb': 'Business',
    'ondernemers': 'Business',
    'zakelijk rijden': 'Business',
    'hr': 'Business',
    'bedrijven': 'Business',
    'zzp': 'Business',

    # charity
    'goededoelen': 'Charity & Social Impact',

    # energy
    'energie': 'Energy & Sustainability',

    # finance
    'verzekering': 'Finance & Investment',
    'bank': 'Finance & Investment',
    'beleggen': 'Finance & Investment',
    'finance': 'Finance & Investment',
    'hypotheek': 'Finance & Investment',
    'centraal beheer': 'Finance & Investment',
    'belastingdienst': 'Finance & Investment',

    # food
    'bezorgmaaltijden': 'Food & Beverages',
    'horeca': 'Food & Beverages',

    # health
    'zorg': 'Health & Wellbeing',

    # lottery
    'kansspelen': 'Lottery & Games',
    'loterij': 'Lottery & Games',
    'winactie': 'Lottery & Games',

    # media
    'bladen': 'Media & Publishing',
    'kranten': 'Media & Publishing',
    'entertainment': 'Media & Publishing',
    'onderzoek': 'Media & Publishing',

    # retail
    'cadeau': 'Retail & Promotion',
    'acties': 'Retail & Promotion',
    'kortingen': 'Retail & Promotion',

    # tech
    'windows': 'Telecom & Technology',
    'it': 'Telecom & Technology',
    'telcom': 'Telecom & Technology',
    'ios': 'Telecom & Technology',
    'android': 'Telecom & Technology',
    'linux': 'Telecom & Technology',
    'chrome os': 'Telecom & Technology',
    'mac os': 'Telecom & Technology',
    'os x': 'Telecom & Technology',


    #other
    'bigevent': 'Other',
    'wetgeving': 'Other',
    'overheid': 'Other',

    #unknown
    'unknown': 'Unknown'
}

In [ ]:
# function to map interest
def map_interest(x):

    # missing values
    if pd.isna(x):
        return 'Missing'
    
    # mapped interest
    return interest_topic_map.get(x, 'Other')


In [ ]:
# apply mapping to interests dataset
df_interests['mapped_interest_topic'] = df_interests['interests'].apply(map_interest)

df_interests[[ 'interests', 'mapped_interest_topic']].head()

In [ ]:
df_interests['mapped_interest_topic'].value_counts(normalize = True)

In [ ]:
# groupby user id, each user with a list of mapped topics
user_interest_topics = (df_interests.groupby('user_id')['mapped_interest_topic'].apply(list).reset_index())

# merge mapped user interests into final topic dataset
df_final_topic = df_final_topic.merge(
    user_interest_topics,
    on = 'user_id',
    how = 'left')

In [ ]:
df_final_topic['mapped_interest_topic'].head()

In [ ]:
# match campaigns with interests function, 1 if campaign topic in user interests, 0 otherwise
def check_interest_match(row):
    interests = row['mapped_interest_topic']

    if not isinstance(interests, list):
        return 0
    
    return 1 if row['main_topic'] in interests else 0

df_final_topic['interest_topic_match'] = df_final_topic.apply(check_interest_match, axis = 1)

In [ ]:
df_final_topic['interest_topic_match'].value_counts(normalize=True)

80.7% of interactions are matches, meaning most campaigns are sent to users who have a declared interest in that topic. Only 19.3% are mismatches.

In [ ]:
# engagement rate on interest match
df_final_topic.groupby('interest_topic_match')[['open', 'click']].mean()

## Subject line & Preheader Analysis

### Subject line Analysis

In [ ]:
df_content = df_final_topic.copy()

In [ ]:
df_content.columns

#### Length

In [ ]:
df_subject = df_content.copy()

# remove missing subject lines
df_subject = df_subject[df_subject['subject_line'].notna()]

# remove empty strings
df_subject = df_subject[df_subject['subject_line'].str.strip() != '']

In [ ]:
# length of subject lines
df_subject['subject_length'] = (df_subject['subject_line'].str.len())

In [ ]:
df_subject['subject_length'].describe()

In [ ]:
# create length buckets
df_subject['subject_length_group'] = pd.cut(
    df_subject['subject_length'],
    bins = [0, 35, 45, 55, 100],
    labels = ['short', 'medium', 'long', 'very_long'])

In [ ]:
# engagement rates based on length
df_subject.groupby('subject_length_group')[['open', 'click']].mean()

In [ ]:
df_subject['subject_length_group'].value_counts(normalize = True)

#### Contain numbers?

In [ ]:
# check if contain numbers
df_subject['subject_contains_number'] = (df_subject['subject_line'].str.contains(r'\d', regex = True))

In [ ]:
# engagement rate
df_subject.groupby('subject_contains_number')[['open', 'click']].mean()

In [ ]:
df_subject['subject_contains_number'].value_counts(normalize = True)

#### Contain exclamation marks?

In [ ]:
# check if contain exclamation marks
df_subject['subject_contains_exclamation'] = (df_subject['subject_line'].str.contains('!', regex = False))

In [ ]:
# engagement rate
df_subject.groupby('subject_contains_exclamation')[['open', 'click']].mean()

In [ ]:
df_subject['subject_contains_exclamation'].value_counts(normalize = True)

#### Contain promotional words?

In [ ]:
promotion_words = [
    'korting',
    'gratis',
    'cadeau',
    'aanbieding',
    'deal',
    'voordeel',
    'win',
    'actie',
    'bonus',
    'besparen'
]

In [ ]:
# check if contain promotion words
promotion_pattern = '|'.join(promotion_words)

df_subject['subject_contains_promotion'] = (df_subject['subject_line'].str.lower().str.contains(promotion_pattern, regex = True))

In [ ]:
# engagement rate
df_subject.groupby('subject_contains_promotion')[['open', 'click']].mean()

In [ ]:
df_subject['subject_contains_promotion'].value_counts(normalize = True)

### Preheader Analysis

In [ ]:
df_preheader = df_final_topic.copy()

# remove missing values
df_preheader = df_preheader[df_preheader['preheader'].notna()]

# remove empty string
df_preheader = df_preheader[df_preheader['preheader'].str.strip() != '']

#### Length

In [ ]:
# length of preheader
df_preheader['preheader_length'] = (df_preheader['preheader'].str.len())

In [ ]:
df_preheader['preheader_length'].describe()

In [ ]:
# create length buckets
df_preheader['preheader_length_group'] = pd.cut(
    df_preheader['preheader_length'],
    bins =[0, 45, 55, 65, 100],
    labels = ['short', 'medium', 'long', 'very_long'])

In [ ]:
# engagement rate based on length
df_preheader.groupby('preheader_length_group')[['open', 'click']].mean()

In [ ]:
df_preheader['preheader_length_group'].value_counts(normalize = True)

#### Contain numbers?

In [ ]:
# check if contain numbers
df_preheader['preheader_contains_number'] = (df_subject['preheader'].str.contains(r'\d', regex = True))

In [ ]:
# engagement rate
df_preheader.groupby('preheader_contains_number')[['open', 'click']].mean()

In [ ]:
df_preheader['preheader_contains_number'].value_counts(normalize = True)

#### Contain exclamation marks?

In [ ]:
# check if contain exclamation marks
df_preheader['preheader_contains_exclamation'] = (df_preheader['preheader'].str.contains('!', regex = False))

In [ ]:
# engagement rate
df_preheader.groupby('preheader_contains_exclamation')[['open', 'click']].mean()

In [ ]:
df_preheader['preheader_contains_exclamation'].value_counts(normalize = True)

#### Contain promotional words?

In [ ]:
# check if contain promotion words
df_preheader['preheader_contains_promotion'] = (df_preheader['preheader'].str.lower().str.contains(promotion_pattern, regex = True))

In [ ]:
# engagement rate
df_preheader.groupby('preheader_contains_promotion')[['open', 'click']].mean()

In [ ]:
df_preheader['preheader_contains_promotion'].value_counts(normalize = True)

## Historical engagement

In [ ]:
# sort mailing id by timeline (proxy)
df_history = df_final_topic.sort_values(['user_id', 'mailing_id']).copy()

In [ ]:
# past cummulative opens (how many opens before this email?)
df_history['past_opens'] = (df_history.groupby('user_id')['open'].cumsum() - df_history['open'])

# past cummulative clicks (how many clicks before this email?)
df_history['past_clicks'] = (df_history.groupby('user_id')['click'].cumsum() - df_history['click'])

# cummulative email received before current email
df_history['emails_before'] = (df_history.groupby('user_id').cumcount())

In [ ]:
# past open rate (average open rate for all email received before current one)
df_history['past_open_rate'] = (df_history['past_opens']/df_history['emails_before'])

# past click rate (average click rate for all email received before current one)
df_history['past_click_rate'] = (df_history['past_clicks']/df_history['emails_before'])

In [ ]:
df_history[['user_id', 'emails_before', 'past_opens', 'past_clicks', 'past_open_rate', 'past_click_rate']].head()

In [ ]:
df_history['past_open_rate'].describe()

In [ ]:
print(df_history['past_click_rate'].describe())
print('\n', df_history['past_click_rate'].quantile([0.9, 0.95, 0.99, 0.999]))

### Historical opens

In [ ]:
# create buckets for past open rate
df_history['past_open_bucket'] = pd.cut(
    df_history['past_open_rate'],
    bins = [0, 0.2, 0.4, 0.6, 0.8, 1],
    labels = [
        'very_low',
        'low',
        'medium',
        'high',
        'very_high'], include_lowest = True)

In [ ]:
df_history['past_open_bucket'].value_counts(normalize = True)

In [ ]:
# compare past open rates with current engagement rates
df_history.groupby('past_open_bucket')[['open', 'click']].mean()

In [ ]:
# emails received before by each bucket
df_history.groupby('past_open_bucket')['emails_before'].mean()

### Historical clicks

In [ ]:
# create binary groups for past click rates
df_history['clicked_before'] = (df_history['past_click_rate'] > 0)

In [ ]:
df_history['clicked_before'].value_counts(normalize = True)

In [ ]:
# compare past click rates with current engagement rates
df_history.groupby('clicked_before')[['open', 'click']].mean()

In [ ]:
# emails received before by each bucket
df_history.groupby('clicked_before')['emails_before'].mean()

### Historical exposure

In [ ]:
print(df_history['emails_before'].describe())
print('\n',df_history['emails_before'].quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

In [ ]:
# create exposure buckets for number of emails received before 
df_history['exposure_bucket'] = pd.cut(
    df_history['emails_before'],
    bins = [0, 5, 15, 30, 50, 120],
    labels = [
        'very_little',
        'little',
        'moderate',
        'heavy',
        'very_heavy'], include_lowest = True)

In [ ]:
df_history['exposure_bucket'].value_counts(normalize = True)

In [ ]:
# engagement rate
df_history.groupby('exposure_bucket')[['open', 'click']].mean()

### Historical opens x Historical exposure

In [ ]:
df_history.groupby(['past_open_bucket', 'exposure_bucket'])[['open', 'click']].mean()

In [ ]:
# pivot open rates
open_heatmap = df_history.pivot_table(
    values = 'open',
    index = 'past_open_bucket',
    columns = 'exposure_bucket',
    aggfunc = 'mean')

open_heatmap


In [ ]:
# heat map

#relabel rows and columns
open_heatmap = open_heatmap.rename(
    index = {
        'very_low': 'Very Low',
        'low': 'Low',
        'medium': 'Moderate',
        'high': 'High',
        'very_high': 'Very High'},

    columns = {
        'very_little': 'Very Low',
        'little': 'Low',
        'moderate': 'Moderate',
        'heavy': 'High',
        'very_heavy': 'Very High'})

# plot
plt.figure(figsize = (10,6))

sns.heatmap(
    open_heatmap,
    annot = True,
    fmt = ".3f",
    cmap = "YlGnBu"
)

plt.title('Future Open Rate by Historical Opens and Exposure')
plt.ylabel('Historical Open Level')
plt.xlabel('Historical Exposure Level')

plt.show()

### Historical clicks x Historical exposure

In [ ]:
df_history.groupby(['exposure_bucket', 'clicked_before'])[['open', 'click']].mean()

In [ ]:
# pivot click rates
click_pivot = df_history.pivot_table(
    values = 'click',
    index = 'exposure_bucket',
    columns = 'clicked_before',
    aggfunc = 'mean')

click_pivot

In [ ]:
interaction_plot_click = (df_history.groupby(['exposure_bucket', 'clicked_before'])[['click']].mean().reset_index())

In [ ]:
# line plot

# relabel x axis
""" 
interaction_plot_click['exposure_bucket'] = interaction_plot_click['exposure_bucket'].replace({
    'very_little': 'Very Low',
    'little': 'Low',
    'moderate': 'Moderate',
    'heavy': 'High',
    'very_heavy': 'Very High'})

"""

# relabel x axis
interaction_plot_click['exposure_bucket'] = interaction_plot_click['exposure_bucket'].cat.rename_categories({
    'very_little': 'Very Low',
    'little': 'Low',
    'moderate': 'Moderate',
    'heavy': 'High',
    'very_heavy': 'Very High'})

# relabel hue
interaction_plot_click['clicked_before'] = interaction_plot_click['clicked_before'].replace({
    True: 'Previously Clicked',
    False: 'Never Clicked Before'})

#plot
plt.figure(figsize=(10,6))

sns.lineplot(
    data=interaction_plot_click,
    x='exposure_bucket',
    y='click',
    hue='clicked_before',
    marker='o'
)

plt.title('Future Click Rate by Historical Click Behavior and Exposure')
plt.xlabel('Historical Exposure Level')
plt.ylabel('Future Click Rate')

plt.legend(title = 'Historical Click Behaviour')

plt.show()

In [ ]:
# pivot open rates
open_pivot = df_history.pivot_table(
    values = 'open',
    index = 'exposure_bucket',
    columns = 'clicked_before',
    aggfunc = 'mean')

open_pivot

In [ ]:
interaction_plot_open = (df_history.groupby(['exposure_bucket', 'clicked_before'])[['open']].mean().reset_index())

In [ ]:
# line plot

# relabel x axis
interaction_plot_open['exposure_bucket'] = interaction_plot_open['exposure_bucket'].cat.rename_categories({
    'very_little': 'Very Low',
    'little': 'Low',
    'moderate': 'Moderate',
    'heavy': 'High',
    'very_heavy': 'Very High'})

# relabel hue
interaction_plot_open['clicked_before'] = interaction_plot_open['clicked_before'].replace({
    True: 'Previously Clicked',
    False: 'Never Clicked Before'})

#plot
plt.figure(figsize=(10,6))

sns.lineplot(
    data=interaction_plot_open,
    x='exposure_bucket',
    y='open',
    hue='clicked_before',
    marker='o'
)

plt.title('Future Open Rate by Historical Click Behavior and Exposure')
plt.xlabel('Historical Exposure Level')
plt.ylabel('Future Open Rate')

plt.legend(title = 'Historical Click Behaviour')


plt.show()

### Historical opens x Historical clicks

In [ ]:
df_history.groupby(['past_open_bucket', 'clicked_before'])[['open', 'click']].mean()

In [ ]:
# create binary open groups (high/low past opens)
df_history['open_behaviour'] = np.where(
    df_history['past_open_rate'] >= df_history['past_open_rate'].median(),      # median as threshold
    'High Historical Opens',
    'Low Historical Opens')

In [ ]:
df_history['open_behaviour'].value_counts(normalize = True)

In [ ]:
# create behavioural groups based on open groups and click groups
conditions = [
    (df_history['open_behaviour'] == 'High Historical Opens') &
    (df_history['clicked_before'] == False),

    (df_history['open_behaviour'] == 'High Historical Opens') &
    (df_history['clicked_before'] == True),

    (df_history['open_behaviour'] == 'Low Historical Opens') &
    (df_history['clicked_before'] == False),

    (df_history['open_behaviour'] == 'Low Historical Opens') &
    (df_history['clicked_before'] == True)]

# group names
choices = [
    'Habitual Openers',         # high past opens, never clicked
    'Highly Engaged Users',     # high past opens, clicked before
    'Disengaged Users',         # low past opens, never clicked
    'Selective Users']          # low past opens, clicked before

df_history['engagement_group'] = np.select(conditions, choices, default = 'Other')

In [ ]:
df_history.groupby('engagement_group')[['open', 'click']].mean().sort_values('click', ascending = False)

In [ ]:
df_history['engagement_group'].value_counts(normalize = True)

## Click-given-open Analysis

In [ ]:
# keep only emails that were open
df_opened = df_history[df_history['open'] == 1].copy()

#click rate a mong opened emails
overall_click_given_open = df_opened['click'].mean()

overall_click_given_open

In [ ]:
df_opened.shape

### Click-given-open by topic

In [ ]:
# click|open by topic
#ranks topics by their post-open conversion rate
df_opened.groupby('main_topic')['click'].mean().sort_values(ascending = False)

In [ ]:
#opened emails per topic
df_opened['main_topic'].value_counts()

In [ ]:
df_opened['main_topic'].value_counts(normalize = True)

### Click-given-open by historical engagement

In [ ]:
# click|open by historical engagement
df_opened.groupby('past_open_bucket')['click'].mean()

In [ ]:
df_opened['past_open_bucket'].value_counts(normalize = True)

### Click-given-open by exposure bucket

In [ ]:
# click|open by exposure bucket
df_opened.groupby('exposure_bucket')['click'].mean()

In [ ]:
df_opened['exposure_bucket'].value_counts(normalize = True)

### Click-given-open by interest match

In [ ]:
df_opened.columns

In [ ]:
# click|open by interest match
df_opened.groupby('interest_topic_match')['click'].mean()

In [ ]:
df_opened['interest_topic_match'].value_counts(normalize = True)

# Analysis Part 3


## Data device, browsers, emailclient

In [ ]:
df_device = pd.read_excel('Data device, browsers, emailclient.xlsx')

In [ ]:
print(df_device.shape)
print(df_device.columns.tolist())


In [ ]:
print(df_device.head(5))
print(df_device.dtypes)

In [ ]:
#different header rows
for i in range(10):
    df_test = pd.read_excel('Data device, browsers, emailclient.xlsx', header=i)
    if not df_test.columns.str.contains('Unnamed').all():
        print(f"Header row: {i}")
        print(df_test.columns.tolist())
        print(df_test.head(3))
        break

#checking all sheets
xl = pd.ExcelFile('Data device, browsers, emailclient.xlsx')
print(f"\nSheet names: {xl.sheet_names}")

In [ ]:
#first 20 rows
df_raw = pd.read_excel('Data device, browsers, emailclient.xlsx', 
                        sheet_name='Blad1',
                        header=None)

print(df_raw.shape)
print(df_raw.to_string()) 

In [ ]:
#rows
for idx, row in df_raw.iterrows():
    if not (pd.isna(row[0]) and pd.isna(row[1])):
        print(f"Row {idx}: {row[0]} | {row[1]}")

In [ ]:
#parsing into three clean dataframes
device_df = pd.DataFrame({
    'category': ['PC', 'Smartphone', 'Tablet', 'Other'],
    'proportion': [0.10, 0.07, 0.01, 0.82]
})

email_client_df = pd.DataFrame({
    'category': ['Gmail', 'Apple Mail', 'Outlook', 'Windows Live Mail', 'Other'],
    'proportion': [0.35, 0.18, 0.09, 0.02, 0.36]
})

browser_df = pd.DataFrame({
    'category': ['Chrome', 'Microsoft Edge', 'Firefox', 'Internet Explorer', 'Other'],
    'proportion': [0.72, 0.23, 0.02, 0.02, 0.005]
})

print(device_df)
print('\n', email_client_df)
print('\n', browser_df)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

data = [
    (device_df, 'Device Distribution'),
    (email_client_df, 'Email Client Distribution'),
    (browser_df, 'Browser Distribution')
]

colors_list = [
    ['#3498db', '#2980b9', '#1abc9c', '#95a5a6'],
    ['#e74c3c', '#2ecc71', '#3498db', '#f39c12', '#95a5a6'],
    ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#95a5a6']
]

for ax, (df, title), colors in zip(axes, data, colors_list):
    wedges, texts, autotexts = ax.pie(
        df['proportion'],
        autopct=lambda p: f'{p:.1f}%' if p > 3 else '',
        colors=colors,
        startangle=90,
        pctdistance=0.75,
    )
    
    for autotext in autotexts:
        autotext.set_fontweight('bold')
        autotext.set_fontsize(10)

    # Use legend instead of labels
    ax.legend(
        wedges,
        df['category'],
        loc='lower center',
        bbox_to_anchor=(0.5, -0.15),
        ncol=2,
        fontsize=9
    )
    ax.set_title(title, fontsize=12, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

## Dataset with time

 ### Dataset

In [ ]:
#check what columns exist
df_final.columns.tolist()

In [ ]:
df_time = pd.read_excel('UVA Robin .xlsx') 

In [ ]:
print(df_time.shape)
print(df_time.columns.tolist())

In [ ]:
print(df_time.head(5))

In [ ]:
df_time.columns = ['client', 'subject_line', 'mailing_id', 'day_of_week',
    'send_date', 'client_bureau', 'campaign_name',
    'log_field', 'target_interests', 'mail_content', 'extra']
df_time.head(5)

In [ ]:
#for each mailing_id, find the row containing 'adressen' and extract from there
def extract_from_log(df_raw, mailing_col='mailing_id'):
    results = []
    df_raw['mailing_id_filled'] = df_raw['mailing_id'].ffill()
    
    for mid, group in df_raw.groupby('mailing_id_filled'):
        log_rows = group[group['log_field'].str.contains('adressen', na=False, case=False)]
        send_hour = None
        audience_size = None
        
        for _, row in log_rows.iterrows():
            log = str(row['log_field'])
            hour_m = re.search(r'\b(\d{1,2})u(?:ur)?\b', log)
            size_m = re.search(r'([\d\.]+)\s+adressen', log, re.IGNORECASE)
            if hour_m:
                send_hour = int(hour_m.group(1))
            if size_m:
                try:
                    audience_size = int(size_m.group(1).replace('.', ''))
                except ValueError:
                    pass
        
        results.append({'mailing_id': mid, 'send_hour': send_hour, 'audience_size': audience_size})
    
    return pd.DataFrame(results)

df_log_info = extract_from_log(df_raw=df_time)

In [ ]:
df_log_info

In [ ]:
# Forward fill day_of_week and send_date to fill gaps
df_time_filled = df_time.copy()
df_time_filled['day_of_week'] = df_time_filled['day_of_week'].ffill()
df_time_filled['send_date'] = df_time_filled['send_date'].ffill()

#keep only rows with valid numeric mailing_id
df_time_filled = df_time_filled[
    df_time_filled['mailing_id'].astype(str).str.match(r'^\d+$')
].copy()
df_time_filled['mailing_id'] = df_time_filled['mailing_id'].astype(int)

# Keep one row per mailing_id (first occurrence)
df_time_clean = df_time_filled.drop_duplicates(subset='mailing_id', keep='first').copy()

# translate Dutch day names to English
day_map = {
    'Maandag': 'Monday', 'Dinsdag': 'Tuesday', 'Woensdag': 'Wednesday',
    'Donderdag': 'Thursday', 'Vrijdag': 'Friday',
    'Zaterdag': 'Saturday', 'Zondag': 'Sunday'
}
df_time_clean['day_of_week'] = df_time_clean['day_of_week'].map(day_map).fillna(df_time_clean['day_of_week'])

print(f"Shape: {df_time_clean.shape}")

In [ ]:
print(f"Unique mailing IDs: {df_time_clean['mailing_id'].nunique()}")
print('\n', df_time_clean['day_of_week'].value_counts(dropna=False))
print('\n', df_time_clean[['mailing_id', 'day_of_week', 'send_date']].head(10))

#### Merge

In [ ]:
#clean df_log_info and df_time_clean
df_log_info_clean = df_log_info[
    df_log_info['mailing_id'].astype(str).str.match(r'^\d+$')].copy()

df_log_info_clean['mailing_id'] = df_log_info_clean['mailing_id'].astype(int)

In [ ]:
#merge
df_time_clean = df_time_clean.merge(df_log_info_clean, on='mailing_id', how='left')

print(f"send_hour NaNs: {df_time_clean['send_hour'].isna().sum()} out of {len(df_time_clean)}")
print(df_time_clean[['mailing_id', 'day_of_week', 'send_date', 'send_hour', 'audience_size']].head(10))

In [ ]:
#merge into df_final
df_final_time = df_final.merge(
    df_time_clean[['mailing_id', 'day_of_week', 'send_date', 'send_hour', 'audience_size']],
    on='mailing_id',
    how='left'
)

df_final_time = df_final_time.merge(
    df_time_clean[['mailing_id', 'client_bureau', 'campaign_name']],
    on='mailing_id', how='left'
)
print(f"Shape: {df_final_time.shape}")

In [ ]:
print(f"day_of_week NaNs: {df_final_time['day_of_week'].isna().sum()}")
print(f"send_hour NaNs: {df_final_time['send_hour'].isna().sum()}")
print(f"Rows WITH day_of_week: {df_final_time['day_of_week'].notna().sum()}")

In [ ]:
print(df_final_time['day_of_week'].value_counts(dropna=False))

In [ ]:
#Coverage
timed_rows = df_final_time['day_of_week'].notna().mean() * 100
n_timed_mail = df_final_time[df_final_time['day_of_week'].notna()]['mailing_id'].nunique()

print(f"df_final unique mailings    : {df_final['mailing_id'].nunique()}")
print(f"df_time_clean unique mailings: {df_time_clean['mailing_id'].nunique()}")
print(f"Mailings matched            : {n_timed_mail}")
print(f"Row coverage (open/click)   : {timed_rows:.1f}%")

df_final_time[['user_id', 'mailing_id', 'open', 'click',
               'day_of_week', 'send_hour', 'audience_size']].head(10)

In [ ]:
total = len(df_final_time)
dow_coverage = df_final_time['day_of_week'].notna().sum()
hour_coverage = df_final_time['send_hour'].notna().sum()
date_coverage = df_final_time['send_date'].notna().sum()

print(f"Total rows          : {total:,}")
print(f"day_of_week coverage: {dow_coverage:,} ({dow_coverage/total*100:.1f}%)")
print(f"send_hour coverage  : {hour_coverage:,} ({hour_coverage/total*100:.1f}%)")
print(f"send_date coverage  : {date_coverage:,} ({date_coverage/total*100:.1f}%)")

Temporal data was available for 186 out of 274 unique campaigns, covering 70% of user-mailing interactions. Analysis is conducted on this subset. Older campaigns (mailing IDs < 1334) lack temporal metadata and are excluded from this section. Findings should be interpreted as indicative rather than exhaustive.

In [ ]:
print(df_final_time[['open', 'click']].isna().sum())

### Send Timing Analysis

#### Day of Week

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

day_stats = (
    df_final_time.dropna(subset=['day_of_week', 'open', 'click'])
    .groupby('day_of_week')
    .agg(open_rate=('open', 'mean'),
         click_rate=('click', 'mean'),
         n_records=('open', 'count'))
    .reindex([d for d in day_order if d in df_final_time['day_of_week'].dropna().unique()])
    .reset_index()
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, metric, label, color in zip(
    axes,
    ['open_rate', 'click_rate'],
    ['Open Rate', 'Click Rate'],
    ['steelblue', 'coral'],
):
    ax.bar(day_stats['day_of_week'], day_stats[metric] * 100, color=color, alpha=0.85)
    ax.set_title(f'{label} by Day of Week')
    ax.set_xlabel('Day of Week')
    ax.set_ylabel(f'{label} (%)')
    ax.tick_params(axis='x', rotation=30)
    
    # Put n= labels INSIDE the bars
    for i, (_, row) in enumerate(day_stats.iterrows()):
        ax.text(i, row[metric] * 100 * 0.5,  # halfway up the bar
                f"n={row['n_records']:,}", 
                ha='center', va='center',
                fontsize=7, color='white', fontweight='bold')
    
    # Add 10% headroom above tallest bar
    ax.set_ylim(0, day_stats[metric].max() * 100 * 1.1)

plt.tight_layout()
plt.show()

print(day_stats[['day_of_week', 'open_rate', 'click_rate', 'n_records']].to_string(index=False))

#### Send Hour

In [ ]:
hour_stats = (
    df_final_time.dropna(subset=['send_hour', 'open', 'click'])
    .groupby('send_hour')
    .agg(open_rate=('open', 'mean'), click_rate=('click', 'mean'),
         n_mailings=('mailing_id', 'nunique'), n_records=('open', 'count'))
    .reset_index()
    .sort_values('send_hour')
)
hour_stats['send_hour'] = hour_stats['send_hour'].astype(int)

In [ ]:
# Remove impossible hours
hour_stats_clean = hour_stats[hour_stats['send_hour'] <= 23].copy()
print(hour_stats_clean[['send_hour', 'open_rate', 'click_rate', 'n_mailings', 'n_records']])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, metric, label, color in zip(
    axes,
    ['open_rate', 'click_rate'],
    ['Open Rate', 'Click Rate'],
    ['steelblue', 'coral'],
):
    x = range(len(hour_stats_clean))  # use index positions instead of raw hours
    ax.bar(x, hour_stats_clean[metric] * 100, color=color, alpha=0.85, width=0.6)
    ax.set_title(f'{label} by Send Hour')
    ax.set_xlabel('Send Hour (24h)')
    ax.set_ylabel(f'{label} (%)')
    ax.set_xticks(x)
    ax.set_xticklabels(hour_stats_clean['send_hour'].astype(int), rotation=45)
    ax.set_ylim(0, hour_stats_clean[metric].max() * 100 * 1.15)
    
    # n labels inside bars
    for i, (_, row) in enumerate(hour_stats_clean.iterrows()):
        ax.text(i, row[metric] * 100 * 0.5,
                f"n={int(row['n_mailings'])}",
                ha='center', va='center',
                fontsize=7, color='white', fontweight='bold')

plt.tight_layout()
plt.show()
print(hour_stats_clean[['send_hour', 'open_rate', 'click_rate', 'n_mailings', 'n_records']].to_string(index=False))

#### Month Analysis

In [ ]:
# Extract month and year from send_date
df_final_time['send_date'] = pd.to_datetime(df_final_time['send_date'])
df_final_time['month'] = df_final_time['send_date'].dt.month
df_final_time['year'] = df_final_time['send_date'].dt.year
df_final_time['month_year'] = df_final_time['send_date'].dt.to_period('M')

# Monthly engagement
monthly_stats = (
    df_final_time.dropna(subset=['month_year', 'open', 'click'])
    .groupby('month_year')
    .agg(open_rate=('open', 'mean'), click_rate=('click', 'mean'),
         n_mailings=('mailing_id', 'nunique'), n_records=('open', 'count'))
    .reset_index()
    .sort_values('month_year')
)

print(monthly_stats.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

x = range(len(monthly_stats))
x_labels = monthly_stats['month_year'].astype(str)

for ax, metric, label, color in zip(
    axes,
    ['open_rate', 'click_rate'],
    ['Open Rate', 'Click Rate'],
    ['steelblue', 'coral'],
):
    ax.plot(x, monthly_stats[metric] * 100, color=color, marker='o', linewidth=2)
    ax.fill_between(x, monthly_stats[metric] * 100, alpha=0.2, color=color)
    ax.set_title(f'{label} over Time (Monthly)')
    ax.set_ylabel(f'{label} (%)')
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, rotation=45)
    ax.set_ylim(0, monthly_stats[metric].max() * 100 * 1.15)
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    
    # Add n= labels
    for i, (_, row) in enumerate(monthly_stats.iterrows()):
        ax.text(i, row[metric] * 100 + row[metric] * 100 * 0.05,
                f"n={int(row['n_mailings'])}",
                ha='center', fontsize=7)

plt.tight_layout()
plt.show()

### Audience Size vs Engagement

In [ ]:
#Groups campaigns into 5 meaningful audience size buckets
size_bins = [0, 10_000, 50_000, 100_000, 300_000, float('inf')]
size_labels = ['<10k', '10k–50k', '50k–100k', '100k–300k', '>300k']
pd.cut(df_final_time['audience_size'], bins=size_bins, labels=size_labels)

In [ ]:
df_final_time['audience_bucket'] = pd.cut(
    df_final_time['audience_size'],
    bins=[0, 10_000, 50_000, 100_000, 300_000, float('inf')],
    labels=['<10k', '10k-50k', '50k-100k', '100k-300k', '>300k']
)

size_stats = (
    df_final_time.dropna(subset=['audience_bucket', 'open', 'click'])
    .groupby('audience_bucket', observed=True)
    .agg(open_rate=('open', 'mean'), click_rate=('click', 'mean'),
         n_mailings=('mailing_id', 'nunique'), n_records=('open', 'count'))
    .reset_index()
)

print(size_stats)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors = ['#2ecc71', '#9b59b6'] 

for ax, metric, label, color in zip(
    axes,
    ['open_rate', 'click_rate'],
    ['Open Rate', 'Click Rate'],
    colors,
):
    x = range(len(size_stats))
    ax.bar(x, size_stats[metric] * 100, color=color, alpha=0.85)
    ax.set_title(f'{label} by Audience Size')
    ax.set_xlabel('Audience Size')
    ax.set_ylabel(f'{label} (%)')
    ax.set_xticks(x)
    ax.set_xticklabels(size_stats['audience_bucket'].astype(str), rotation=30)
    ax.set_ylim(0, size_stats[metric].max() * 100 * 1.15)
    
    # Labels inside bars
    for i, (_, row) in enumerate(size_stats.iterrows()):
        ax.text(i, row[metric] * 100 * 0.5,
                f"n={int(row['n_mailings'])}",
                ha='center', va='center',
                fontsize=9, color='white', fontweight='bold')

plt.tight_layout()
plt.show()

### Cross Analysis

#### Open x Topic x Day of the week

In [ ]:
#check what columns exist
df_final_time.columns.tolist()

In [ ]:
#main_topic
df_final_time = df_final_time.merge(
    df_final_topic[['mailing_id', 'main_topic']].drop_duplicates(subset='mailing_id'),
    on='mailing_id',
    how='left')

print(df_final_time['main_topic'].value_counts(dropna=False))

In [ ]:
#rows with day_of_week and main_topic
df_topic_time = df_final_time.dropna(subset=['day_of_week', 'main_topic']).copy()
df_topic_time = df_topic_time[df_topic_time['main_topic'] != 'Missing']

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

#Open rate by day x topic
topic_day_open = (
    df_topic_time.groupby(['main_topic', 'day_of_week'])['open']
    .mean()
    .unstack('day_of_week')
    .reindex(columns=[d for d in day_order if d in df_topic_time['day_of_week'].unique()]))

print(topic_day_open.round(3))

In [ ]:
plt.figure(figsize=(12, 6))
sns.heatmap(
    topic_day_open,
    annot=True,
    fmt='.2f',
    cmap='YlGnBu',
    linewidths=0.5)

plt.title('Open Rate by Topic and Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Topic')
plt.tight_layout()
plt.show()

#### Click x Topic x Day of the week

In [ ]:
# Click rate by day x topic
topic_day_click = (
    df_topic_time.groupby(['main_topic', 'day_of_week'])['click']
    .mean()
    .unstack('day_of_week')
    .reindex(columns=[d for d in day_order if d in df_topic_time['day_of_week'].unique()])
)

print(topic_day_click.round(4))

In [ ]:
plt.figure(figsize=(12, 6))
sns.heatmap(
    topic_day_click,
    annot=True,
    fmt='.4f',
    cmap='YlOrRd',
    linewidths=0.5,
    mask=topic_day_click.isna()
)
plt.title('Click Rate by Topic and Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Topic')
plt.tight_layout()
plt.show()

In [ ]:
#campaigns behind Lottery & Games Friday
df_topic_time[
    (df_topic_time['main_topic'] == 'Lottery & Games') & 
    (df_topic_time['day_of_week'] == 'Friday')
]['mailing_id'].nunique()

# DF_FINAL_TIME is the dataset with the time

## Analysis

In [ ]:
history_cols = set(df_history.columns.tolist())
history_cols

In [ ]:
final_time_cols = set(df_final_time.columns.tolist())
final_time_cols

In [ ]:
#columns
history_cols = set(df_history.columns.tolist())
final_time_cols = set(df_final_time.columns.tolist())

print("In df_history but NOT in df_final_time:")
print(history_cols - final_time_cols)

print("\nIn df_final_time but NOT in df_history:")
print(final_time_cols - history_cols)

In [ ]:
#from df_history that are missing from df_final_time
history_extra_cols = ['user_id', 'mailing_id', 'engagement_group', 'open_segment', 
                      'past_open_bucket', 'clicked_before', 'click_segment', 
                      'open_behaviour', 'interest_topic_match', 'emails_before', 
                      'past_click_rate', 'past_clicks', 'past_open_rate', 
                      'exposure_bucket', 'past_opens', 'mapped_interest_topic']

#merge into df_final_time
df_master = df_final_time.merge(
    df_history[history_extra_cols],
    on=['user_id', 'mailing_id'],
    how='left'
)

print(f"Shape: {df_master.shape}")
print(f"Columns: {len(df_master.columns)}")
print(df_master.columns.tolist())

In [ ]:
missing = df_master.isna().sum()
missing_pct = (missing / len(df_master) * 100).round(1)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print(missing_df[missing_df['missing_count'] > 0].sort_values('missing_pct', ascending=False))

In [ ]:
print(df_master.dtypes)

In [ ]:
#all numeric columns in df_master
numeric_cols = df_master.select_dtypes(include='number').columns.tolist()
print(numeric_cols)

In [ ]:
feature_cols = [
    # User profile features
    'age', 
    'n_interests',
    # Historical behaviour features
    'past_open_rate',
    'past_click_rate', 
    'past_opens',
    'past_clicks',
    'emails_before',
    # Interest matching
    'interest_topic_match',
    # Temporal features
    'send_hour',
    'month',
    # Campaign features
    'audience_size',]

corr_df = df_master[feature_cols + ['open', 'click']].corr()[['open', 'click']].drop(['open', 'click'])

print(corr_df.sort_values('open', ascending=False).round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 8))
sns.heatmap(
    corr_df.sort_values('open', ascending=False),
    annot=True,
    fmt='.3f',
    cmap='RdYlGn',
    center=0,
    vmin=-0.3,
    vmax=0.3,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Feature Correlation with Engagement\n(Pearson correlation)')
plt.tight_layout()
plt.show()

## Dropping unusable columns

In [ ]:
df_model = df_master.copy()

In [ ]:
#Drop columns with >40% missing except for age and send_hour
cols_to_drop = ['city', 'client_bureau', 'campaign_name', 'birth_date', 'audience_size', 'audience_bucket']
df_model = df_model.drop(columns=cols_to_drop, errors='ignore')

In [ ]:
#Drop columns with not usable features
cols_to_drop_1= [ 'month_year', 'send_date', 'age', 'send_hour']
df_model = df_model.drop(columns=cols_to_drop_1, errors='ignore')

In [ ]:
#Impute past_open_rate and past_click_rate with 0 (no history = no opens/clicks)
df_model['past_open_rate'] = df_model['past_open_rate'].fillna(0)
df_model['past_click_rate'] = df_model['past_click_rate'].fillna(0)
df_model['past_opens'] = df_model['past_opens'].fillna(0)
df_model['past_clicks'] = df_model['past_clicks'].fillna(0)

In [ ]:
print(df_model.isnull().sum())
print(df_model.shape)

In [ ]:
#Drop other unusable columns
extra_drop = [ 'mailing_info', 'subject_line', 'preheader', 'past_open_bucket', 'year','postcode', 'engagement_group', 'clicked_before']  
df_model = df_model.drop(columns=extra_drop, errors='ignore')

In [ ]:
#drop day_of_week 
df_model = df_model.drop(columns=['day_of_week'])

#Filling month via distribution sampling
month_dist = df_model['month'].dropna()
df_model['month'] = df_model['month'].fillna(
    pd.Series(
        month_dist.sample(df_model['month'].isna().sum(), 
                         replace=True, 
                         random_state=42).values,
        index=df_model[df_model['month'].isna()].index
    )
)

In [ ]:
print(df_model.isnull().sum())
print(df_model.shape)

In [ ]:
last_drop = [
    'mapped_interest_topic',  # lists - unusable
    'open_segment',           # redundant with past_open_rate
    'click_segment',          # redundant with past_click_rate + clicked_before
    'exposure_bucket',        # redundant with emails_before
    'open_behaviour',         # redundant with past_open_rate
    'interests',              # raw lists - already encoded
]

df_model = df_model.drop(columns=last_drop, errors='ignore')


In [ ]:
print(df_model.shape)
print(df_model.columns.tolist())

In [ ]:
print(df_model['open_type'].value_counts())
print('\n', df_model['click_type'].value_counts())

In [ ]:
df_model = df_model.drop(columns=['open_type', 'click_type'])
print(df_model.shape)
print(df_model.columns.tolist())

## Summary before modeling

In [ ]:
feature_reliability = pd.DataFrame({
    'feature': [
        'past_open_rate', 'past_click_rate', 'past_opens', 'past_clicks',
        'emails_before', 'interest_topic_match', 'n_interests',
        'month', 'gender', 'main_topic'
    ],
    'missing_pct': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    'imputation': [
        '0 (no history = no engagement)',
        '0 (no history = no engagement)',
        '0 (no history = no engagement)',
        '0 (no history = no engagement)',
        '0 (no history = no engagement)',
        'None required',
        'None required',
        'Distribution sampling (preserves seasonal pattern)',
        'None required',
        'None required'
    ],
    'reliability': [
        'High — strongest predictor of opens (r=0.549)',
        'High — strongest predictor of clicks (r=0.449)',
        'High — supports past_open_rate',
        'High — supports past_click_rate',
        'High — captures cumulative exposure history',
        'High — binary flag, directly derived from topic mapping',
        'High — fully observed user profile feature',
        'Medium — imputed for 29.4% of rows via distribution sampling',
        'High — fully observed after cleaning',
        'High — fully observed, 10 categories'
    ]
})

print(feature_reliability.to_string(index=False))

In [ ]:
master_cols = set(df_master.columns.tolist())
model_cols = set(df_model.columns.tolist())

dropped = master_cols - model_cols

print(f"Columns in df_master but NOT in df_model ({len(dropped)} total):")
for col in sorted(dropped):
    print(f"  {col}")

In [ ]:
print("\nExcluded features:")
excluded = {
    # High missingness — unreliable as model inputs
    'city':                  '92.0% missing — unusable',
    'client_bureau':         '87.8% missing — only covers recent campaigns',
    'campaign_name':         '87.8% missing — only covers recent campaigns',
    'birth_date':            '74.3% missing — age derived from this and also excluded',
    'age':                   '74.3% missing — imputing majority of users with median unreliable',
    'send_hour':             '50.0% missing — only 2 send hours observed, insufficient coverage',
    'audience_size':         '43.3% missing — also collinear with main_topic',
    'audience_bucket':       '43.3% missing — derived from audience_size',
    'postcode':              '37.8% missing + high cardinality — encoding not feasible',
    'day_of_week':           '29.4% missing — send-time optimisation handled separately by teammate',

    # Redundant temporal columns
    'send_date':             'Redundant — month extracted and kept separately',
    'month_year':            'Redundant — month extracted and kept separately',
    'year':                  'No meaningful variation across dataset',

    # Text columns — not usable in logistic regression
    'mailing_info':          'Free text — not usable in logistic regression',
    'subject_line':          'Free text — not usable in logistic regression',
    'preheader':             'Free text — not usable in logistic regression',

    # Derived segmentation columns — redundant with source features
    'past_open_bucket':      'Derived from past_open_rate — redundant',
    'open_segment':          'Derived from past_open_rate — redundant',
    'open_behaviour':        'Derived from past_open_rate — redundant',
    'click_segment':         'Derived from past_click_rate — redundant',
    'clicked_before':        'Derived from past_click_rate — redundant',
    'exposure_bucket':       'Derived from emails_before — redundant',
    'engagement_group':      'Derived from past_open_rate + past_click_rate — redundant',

    # Intermediate processing columns
    'open_type':             'Intermediate parsing column — not a feature',
    'click_type':            'Intermediate parsing column — not a feature',

    # List-format columns — captured via derived features
    'interests':             'List format — captured via n_interests',
    'mapped_interest_topic': 'List format — topic alignment captured via interest_topic_match',
}

for feature, reason in excluded.items():
    print(f"  {feature:<25}: {reason}")

In [ ]:
print(df_model[['open', 'click']].isna().sum())
print(f"Total rows: {len(df_model)}")

In [ ]:
print(df_model.isna().sum())

In [ ]:
print('open' in df_exploded.columns)
print('open' in df2.columns)

## Prep

In [ ]:
df_model.dtypes

In [ ]:
#encoding categorical variables using one-hot encoding
df_model = pd.get_dummies(df_model, 
                           columns=['gender', 'main_topic'],
                           drop_first=True)

print(df_model.shape)
print(df_model.columns.tolist())

In [ ]:
df_model = df_model.dropna(subset=['open', 'click'])
print(f"Rows remaining: {len(df_model)}")

In [ ]:
#drop ID columns
df_model = df_model.drop(columns=['user_id', 'mailing_id'])

## Individual Part (Embeddings)

## Testing SBERT word embeddings

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

# load multilingual model
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# real subject lines from the dataset
subject_lines = [
    "Lezen, puzzelen, genieten: 4 edities cadeau",
    "4 weken jouw favoriete krant digitaal voor maar 1,-",
    "Profiteer nu: €300 bonus én vaste energietarieven",
    "Jouw mening is geld waard – start vandaag",
    "Speel mee met de ENGIE woordlegger!",
    "Jouw kans op een Douwe Egberts gevuld koffieblik",
    "Heb je jouw kans op het koffieblik t.w.v. €50?",
    "Altijd iets te lezen, waar je ook bent!",
    "Win een SimpSune powerbank t.w.v. €79,95",
    "Laat je mening horen, ontvang cadeaubonnen",
]

# embed
embeddings = model.encode(subject_lines)

# pairwise cosine similarity
sim_matrix = cosine_similarity(embeddings)

# display as dataframe
labels = [s[:40] for s in subject_lines]
sim_df = pd.DataFrame(sim_matrix, index=labels, columns=labels)

print("=== paraphrase-multilingual-MiniLM-L12-v2 ===")
print(sim_df.round(2).to_string())

In [ ]:
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

pairs = [
    ("Jouw kans op een Douwe Egberts gevuld koffieblik",
     "Heb je jouw kans op het koffieblik t.w.v. €50?"),
    
    ("Lezen, puzzelen, genieten: 4 edities cadeau",
     "Altijd iets te lezen, waar je ook bent!"),
    
    ("Lezen, puzzelen, genieten: 4 edities cadeau",
     "4 weken jouw favoriete krant digitaal voor maar 1,-"),

    ("Jouw mening is geld waard – start vandaag",
     "Laat je mening horen, ontvang cadeaubonnen"),

    ("Lezen, puzzelen, genieten: 4 edities cadeau",
     "Speel mee met de ENGIE woordlegger!"),

    ("Profiteer nu: €300 bonus én vaste energietarieven",
     "Jouw mening is geld waard – start vandaag"),

    ("Profiteer nu: €300 bonus én vaste energietarieven",
     "Altijd iets te lezen, waar je ook bent!"),

    ("Win een SimpSune powerbank t.w.v. €79,95",
     "Altijd iets te lezen, waar je ook bent!"),
]

# compute scores
results = []
for a, b in pairs:
    emb_a = model.encode([a])
    emb_b = model.encode([b])
    score = cosine_similarity(emb_a, emb_b)[0][0]
    results.append({
        'Subject Line A': a,
        'Subject Line B': b,
        'Cosine Similarity': round(float(score), 3)
    })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

## Testing Dutch BERT Embeddings

In [ ]:
# Dutch BERT model
model_nl = SentenceTransformer("wietsedv/bert-base-dutch-cased")

subject_lines = [
    "Lezen, puzzelen, genieten: 4 edities cadeau",
    "4 weken jouw favoriete krant digitaal voor maar 1,-",
    "Profiteer nu: €300 bonus én vaste energietarieven",
    "Jouw mening is geld waard – start vandaag",
    "Speel mee met de ENGIE woordlegger!",
    "Jouw kans op een Douwe Egberts gevuld koffieblik",
    "Heb je jouw kans op het koffieblik t.w.v. €50?",
    "Altijd iets te lezen, waar je ook bent!",
    "Win een SimpSune powerbank t.w.v. €79,95",
    "Laat je mening horen, ontvang cadeaubonnen",
]

embeddings_nl = model_nl.encode(subject_lines)
sim_matrix_nl = cosine_similarity(embeddings_nl)

labels = [s[:40] for s in subject_lines]
sim_df_nl = pd.DataFrame(sim_matrix_nl, index=labels, columns=labels)

print("=== wietsedv/bert-base-dutch-cased ===")
print(sim_df_nl.round(2).to_string())

In [ ]:
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

unique_campaigns = (
    df_master[["mailing_id", "subject_line"]]
    .drop_duplicates("mailing_id")
    .copy()
)
unique_campaigns["subject_line"] = unique_campaigns["subject_line"].fillna("").astype(str)

subject_embeddings = model.encode(
    unique_campaigns["subject_line"].tolist(),
    batch_size=32,
    show_progress_bar=True
)

# mailing_id → embedding
mailing_embed_dict = dict(zip(unique_campaigns["mailing_id"], subject_embeddings))
print(f"Embedded {len(mailing_embed_dict)} unique campaigns ✅")

## M1: Logistic Regression

In [ ]:
results_m1 = []

for fold in range(n_folds):
    test_ids  = unique_mailings[fold * fold_size : (fold+1) * fold_size]
    train_ids = unique_mailings[~np.isin(unique_mailings, test_ids)]

    df_train = df_model[df_model["mailing_id"].isin(train_ids)].copy()
    df_test  = df_model[df_model["mailing_id"].isin(test_ids)].copy()

    y_train = df_train["open"]
    y_test  = df_test["open"]

    # single feature
    X_train_m1 = df_train[["past_open_rate"]].copy()
    X_test_m1  = df_test[["past_open_rate"]].copy()

    scaler_m1     = StandardScaler()
    X_train_m1_sc = scaler_m1.fit_transform(X_train_m1)
    X_test_m1_sc  = scaler_m1.transform(X_test_m1)

    lr_m1 = LogisticRegression(penalty="l2", C=1.0, random_state=42,
                                max_iter=1000, solver="saga")
    lr_m1.fit(X_train_m1_sc, y_train)
    auc_m1 = roc_auc_score(y_test, lr_m1.predict_proba(X_test_m1_sc)[:, 1])
    results_m1.append(auc_m1)
    print(f"Fold {fold+1} M1 AUC: {auc_m1:.4f}")

print(f"\nM1 mean AUC: {np.mean(results_m1):.4f} ± {np.std(results_m1):.4f}")

## M2 and M3: LOGISTIC REGGRESSION

In [ ]:
# Clean feature columns
df_model["mailing_id"] = df_master.loc[df_model.index, "mailing_id"].values
df_model["user_id"]    = df_master.loc[df_model.index, "user_id"].values

feature_cols = [c for c in df_model.columns
                if c not in ["open", "click", "mailing_id", "user_id",
                             "send_date", "semantic_match_score", "novelty_score"]
                and df_model[c].dtype != "object"
                and str(df_model[c].dtype) != "datetime64[ns]"]

feature_cols_no_topic = [c for c in feature_cols 
                         if not c.startswith("main_topic_")
                         and c != "interest_topic_match"]

print(f"Full features ({len(feature_cols)}):           {feature_cols}")
print(f"No topic features ({len(feature_cols_no_topic)}): {feature_cols_no_topic}")

# Fold setup
unique_mailings = df_model["mailing_id"].drop_duplicates().to_numpy()
np.random.seed(42)
np.random.shuffle(unique_mailings)

n_folds   = 5
fold_size = len(unique_mailings) // n_folds
results_m2 = []
results_m3 = []

# Scoring function
def score_rows(df_rows, user_pref, user_recv, mailing_embed_dict):
    sem = np.full(len(df_rows), 0.5)
    nov = np.full(len(df_rows), 0.5)
    for i, (uid, mid) in enumerate(zip(df_rows["user_id"].values,
                                       df_rows["mailing_id"].values)):
        s_emb = mailing_embed_dict.get(mid)
        if s_emb is None:
            continue
        if uid in user_pref:
            sem[i] = float(cosine_similarity([user_pref[uid]], [s_emb])[0][0])
        if uid in user_recv:
            nov[i] = float(1 - cosine_similarity([user_recv[uid]], [s_emb])[0][0])
    return sem, nov

# Cross validation
for fold in range(n_folds):
    print(f"\n{'='*50}")
    print(f"FOLD {fold+1}/5")
    print(f"{'='*50}")

    test_ids  = unique_mailings[fold * fold_size : (fold+1) * fold_size]
    train_ids = unique_mailings[~np.isin(unique_mailings, test_ids)]

    df_train = df_model[df_model["mailing_id"].isin(train_ids)].copy()
    df_test  = df_model[df_model["mailing_id"].isin(test_ids)].copy()

    y_train = df_train["open"]
    y_test  = df_test["open"]

    print(f"Train: {len(df_train):,} rows | Test: {len(df_test):,} rows")
    print(f"Train open rate: {y_train.mean():.3f} | Test open rate: {y_test.mean():.3f}")

    # M2: baseline
    X_train_m2 = df_train[feature_cols].copy()
    X_test_m2  = df_test[feature_cols].copy()

    scaler_m2     = StandardScaler()
    X_train_m2_sc = scaler_m2.fit_transform(X_train_m2)
    X_test_m2_sc  = scaler_m2.transform(X_test_m2)

    lr_m2 = LogisticRegression(penalty="l2", C=1.0, random_state=42,
                                max_iter=1000, solver="saga")
    lr_m2.fit(X_train_m2_sc, y_train)
    auc_m2 = roc_auc_score(y_test, lr_m2.predict_proba(X_test_m2_sc)[:, 1])
    results_m2.append(auc_m2)
    print(f"M2 AUC: {auc_m2:.4f}")

    # build preference and received vectors from train only
    # preference vector — opens only
    opened = df_train[df_train["open"] == 1.0][["user_id", "mailing_id"]].copy()
    opened["emb"] = opened["mailing_id"].map(mailing_embed_dict)
    opened = opened.dropna(subset=["emb"])

    user_pref = {}
    for uid, grp in opened.groupby("user_id"):
        user_pref[uid] = np.mean(np.stack(grp["emb"].values), axis=0)

    # received mean — all train interactions
    received = df_train[["user_id", "mailing_id"]].copy()
    received["emb"] = received["mailing_id"].map(mailing_embed_dict)
    received = received.dropna(subset=["emb"])

    user_recv = {}
    for uid, grp in received.groupby("user_id"):
        user_recv[uid] = np.mean(np.stack(grp["emb"].values), axis=0)

    # compute semantic scores
    print("Computing semantic scores...")
    train_sem, train_nov = score_rows(df_train, user_pref, user_recv, mailing_embed_dict)
    test_sem,  test_nov  = score_rows(df_test,  user_pref, user_recv, mailing_embed_dict)

    # M3: baseline + semantic
    X_train_m3 = df_train[feature_cols].copy()
    X_test_m3  = df_test[feature_cols].copy()

    X_train_m3 = X_train_m3.assign(
        semantic_match_score=train_sem,
        novelty_score=train_nov
    )
    X_test_m3 = X_test_m3.assign(
        semantic_match_score=test_sem,
        novelty_score=test_nov
    )

    scaler_m3     = StandardScaler()
    X_train_m3_sc = scaler_m3.fit_transform(X_train_m3)
    X_test_m3_sc  = scaler_m3.transform(X_test_m3)

    lr_m3 = LogisticRegression(penalty="l2", C=1.0, random_state=42,
                                max_iter=1000, solver="saga")
    lr_m3.fit(X_train_m3_sc, y_train)
    auc_m3 = roc_auc_score(y_test, lr_m3.predict_proba(X_test_m3_sc)[:, 1])
    results_m3.append(auc_m3)
    print(f"M3 AUC: {auc_m3:.4f}")

# Final results
print("\n" + "="*50)
print("FINAL RESULTS — 5-Fold Cross Validation")
print("="*50)
print(f"M2 per fold: {[round(x, 4) for x in results_m2]}")
print(f"M3 per fold: {[round(x, 4) for x in results_m3]}")
print(f"\nM2 mean AUC: {np.mean(results_m2):.4f} ± {np.std(results_m2):.4f}")
print(f"M3 mean AUC: {np.mean(results_m3):.4f} ± {np.std(results_m3):.4f}")
print(f"Improvement: {np.mean(results_m3) - np.mean(results_m2):+.4f}")

## M2 LOGISTIC REGRESSION: COEFFICIENTS WITH P-VALUES

In [ ]:
from scipy import stats
import numpy as np
import pandas as pd

# M2 final model (same split as M3/M4 for direct comparison)

X_train_m2_f = df_train_final[feature_cols].copy()
X_test_m2_f  = df_test_final[feature_cols].copy()

scaler_m2_f     = StandardScaler()
X_train_m2_f_sc = scaler_m2_f.fit_transform(X_train_m2_f)
X_test_m2_f_sc  = scaler_m2_f.transform(X_test_m2_f)

lr_m2_final = LogisticRegression(penalty="l2", C=1.0, random_state=42,
                                  max_iter=1000, solver="saga")
lr_m2_final.fit(X_train_m2_f_sc, y_train_final)

auc_m2_final = roc_auc_score(y_test_final, lr_m2_final.predict_proba(X_test_m2_f_sc)[:, 1])
print(f"M2 final model AUC: {auc_m2_final:.4f}")

# p-values for M2
np.random.seed(42)
sample_idx_m2 = np.random.choice(X_train_m2_f_sc.shape[0], 10000, replace=False)
X_sample_m2   = X_train_m2_f_sc[sample_idx_m2]

y_prob_sample_m2 = lr_m2_final.predict_proba(X_sample_m2)[:, 1]
W_m2 = y_prob_sample_m2 * (1 - y_prob_sample_m2)

X_with_const_m2 = np.hstack([np.ones((X_sample_m2.shape[0], 1)), X_sample_m2])
ridge_m2 = 1e-5 * np.eye(X_with_const_m2.shape[1])
cov_matrix_m2 = np.linalg.inv(X_with_const_m2.T @ (W_m2[:, None] * X_with_const_m2) + ridge_m2)
std_errors_m2 = np.sqrt(np.diag(cov_matrix_m2))[1:]

coefficients_m2 = lr_m2_final.coef_[0]
z_scores_m2     = coefficients_m2 / std_errors_m2
p_values_m2     = 2 * (1 - stats.norm.cdf(np.abs(z_scores_m2)))

results_df_m2 = pd.DataFrame({
    'feature':     feature_cols,
    'coefficient': coefficients_m2,
    'abs_coef':    np.abs(coefficients_m2),
    'std_error':   std_errors_m2,
    'z_score':     z_scores_m2,
    'p_value':     p_values_m2.round(4),
    'significant': p_values_m2 < 0.05
}).sort_values('abs_coef', ascending=False).drop(columns='abs_coef')

print("\n" + "="*60)
print("M2 LOGISTIC REGRESSION — COEFFICIENTS WITH P-VALUES")
print("="*60)
print(results_df_m2.to_string(index=False))

## M2 LOGISTIC REGRESSION: COEFFICIENTS WITH P-VALUES

In [ ]:
np.random.seed(42)
np.random.shuffle(unique_mailings)

final_test_ids  = unique_mailings[:54]   # last fold as holdout
final_train_ids = unique_mailings[54:]   # rest for training

df_train_final = df_model[df_model["mailing_id"].isin(final_train_ids)].copy()
df_test_final  = df_model[df_model["mailing_id"].isin(final_test_ids)].copy()

y_train_final = df_train_final["open"]
y_test_final  = df_test_final["open"]

# Compute semantic scores for final model
opened_final = df_train_final[df_train_final["open"] == 1.0][["user_id", "mailing_id"]].copy()
opened_final["emb"] = opened_final["mailing_id"].map(mailing_embed_dict)
opened_final = opened_final.dropna(subset=["emb"])

user_pref_final = {}
for uid, grp in opened_final.groupby("user_id"):
    user_pref_final[uid] = np.mean(np.stack(grp["emb"].values), axis=0)

received_final = df_train_final[["user_id", "mailing_id"]].copy()
received_final["emb"] = received_final["mailing_id"].map(mailing_embed_dict)
received_final = received_final.dropna(subset=["emb"])

user_recv_final = {}
for uid, grp in received_final.groupby("user_id"):
    user_recv_final[uid] = np.mean(np.stack(grp["emb"].values), axis=0)

train_sem_f, train_nov_f = score_rows(df_train_final, user_pref_final, user_recv_final, mailing_embed_dict)
test_sem_f,  test_nov_f  = score_rows(df_test_final,  user_pref_final, user_recv_final, mailing_embed_dict)

# Build feature matrices
X_train_f = df_train_final[feature_cols].copy().assign(
    semantic_match_score=train_sem_f,
    novelty_score=train_nov_f
)
X_test_f = df_test_final[feature_cols].copy().assign(
    semantic_match_score=test_sem_f,
    novelty_score=test_nov_f
)

# Scale and train
scaler_f = StandardScaler()
X_train_f_sc = scaler_f.fit_transform(X_train_f)
X_test_f_sc  = scaler_f.transform(X_test_f)

lr_final = LogisticRegression(penalty="l2", C=1.0, random_state=42,
                               max_iter=1000, solver="saga")
lr_final.fit(X_train_f_sc, y_train_f := y_train_final)

print(f"Final model AUC: {roc_auc_score(y_test_final, lr_final.predict_proba(X_test_f_sc)[:,1]):.4f}")

# Compute p-values
np.random.seed(42)
sample_idx = np.random.choice(X_train_f_sc.shape[0], 10000, replace=False)
X_sample   = X_train_f_sc[sample_idx]

y_prob_sample = lr_final.predict_proba(X_sample)[:, 1]
W = y_prob_sample * (1 - y_prob_sample)

X_with_const = np.hstack([np.ones((X_sample.shape[0], 1)), X_sample])
ridge = 1e-5 * np.eye(X_with_const.shape[1])
cov_matrix = np.linalg.inv(X_with_const.T @ (W[:, None] * X_with_const) + ridge)
std_errors  = np.sqrt(np.diag(cov_matrix))[1:]

coefficients = lr_final.coef_[0]
z_scores     = coefficients / std_errors
p_values     = 2 * (1 - stats.norm.cdf(np.abs(z_scores)))

# Results table
feature_names = list(feature_cols) + ['semantic_match_score', 'novelty_score']

results_df = pd.DataFrame({
    'feature':     feature_names,
    'coefficient': coefficients,
    'abs_coef':    np.abs(coefficients),
    'std_error':   std_errors,
    'z_score':     z_scores,
    'p_value':     p_values.round(4),
    'significant': p_values < 0.05
}).sort_values('abs_coef', ascending=False).drop(columns='abs_coef')

print("\n" + "="*60)
print("M3 LOGISTIC REGRESSION — COEFFICIENTS WITH P-VALUES")
print("(estimated from 10,000 row sample, p-values approximate)")
print("="*60)
print(results_df.to_string(index=False))

## M4: Random Forest

In [ ]:
# M4: Random Forest - all 5 folds
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

results_m4 = []

for fold in range(n_folds):
    print(f"\nFOLD {fold+1}/5")

    test_ids  = unique_mailings[fold * fold_size : (fold+1) * fold_size]
    train_ids = unique_mailings[~np.isin(unique_mailings, test_ids)]

    df_train = df_model[df_model["mailing_id"].isin(train_ids)].copy()
    df_test  = df_model[df_model["mailing_id"].isin(test_ids)].copy()

    y_train = df_train["open"]
    y_test  = df_test["open"]

    # recompute semantic scores for this fold
    opened = df_train[df_train["open"] == 1.0][["user_id", "mailing_id"]].copy()
    opened["emb"] = opened["mailing_id"].map(mailing_embed_dict)
    opened = opened.dropna(subset=["emb"])

    user_pref = {}
    for uid, grp in opened.groupby("user_id"):
        user_pref[uid] = np.mean(np.stack(grp["emb"].values), axis=0)

    received = df_train[["user_id", "mailing_id"]].copy()
    received["emb"] = received["mailing_id"].map(mailing_embed_dict)
    received = received.dropna(subset=["emb"])

    user_recv = {}
    for uid, grp in received.groupby("user_id"):
        user_recv[uid] = np.mean(np.stack(grp["emb"].values), axis=0)

    train_sem, train_nov = score_rows(df_train, user_pref, user_recv, mailing_embed_dict)
    test_sem,  test_nov  = score_rows(df_test,  user_pref, user_recv, mailing_embed_dict)

    X_train_m4 = df_train[feature_cols].copy().assign(
        semantic_match_score=train_sem,
        novelty_score=train_nov
    )
    X_test_m4 = df_test[feature_cols].copy().assign(
        semantic_match_score=test_sem,
        novelty_score=test_nov
    )

    rf_m4 = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
    rf_m4.fit(X_train_m4, y_train)
    auc_m4 = roc_auc_score(y_test, rf_m4.predict_proba(X_test_m4)[:, 1])
    results_m4.append(auc_m4)
    print(f"M4 AUC: {auc_m4:.4f}")

print("\n" + "="*50)
print(f"M4 per fold: {[round(x, 4) for x in results_m4]}")
print(f"M4 mean AUC: {np.mean(results_m4):.4f} ± {np.std(results_m4):.4f}")
print(f"M4 vs M2:    {np.mean(results_m4) - np.mean(results_m2):+.4f}")
print(f"M4 vs M3:    {np.mean(results_m4) - np.mean(results_m3):+.4f}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Train final RF model on same split
rf_final = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
rf_final.fit(X_train_f, y_train_final)

print(f"Final RF AUC: {roc_auc_score(y_test_final, rf_final.predict_proba(X_test_f)[:,1]):.4f}")

# Feature importance
feature_names = list(feature_cols) + ['semantic_match_score', 'novelty_score']

importance_df = pd.DataFrame({
    'feature':    feature_names,
    'importance': rf_final.feature_importances_
}).sort_values('importance', ascending=False)

importance_df['importance_pct'] = (importance_df['importance'] * 100).round(2)
importance_df['cumulative_pct'] = importance_df['importance_pct'].cumsum().round(2)

print("\n" + "="*60)
print("M4 RANDOM FOREST — FEATURE IMPORTANCE (Gini)")
print("="*60)
print(importance_df.to_string(index=False))

import matplotlib.pyplot as plt

top15 = importance_df.head(15)
colors = ['#1F4E79' if f in ['semantic_match_score', 'novelty_score'] 
          else '#2E75B6' if f in ['past_open_rate', 'past_opens', 'past_clicks', 
                                   'past_click_rate', 'emails_before']
          else '#AAAAAA' for f in top15['feature']]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top15['feature'][::-1], top15['importance_pct'][::-1], color=colors[::-1])
ax.set_xlabel('Feature importance (%)')
ax.set_title('M4 Random Forest — Top 15 feature importances')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#1F4E79', label='Semantic features'),
    Patch(facecolor='#2E75B6', label='Behavioral features'),
    Patch(facecolor='#AAAAAA', label='Other features')
]
ax.legend(handles=legend_elements, loc='lower right')
plt.tight_layout()
plt.show()

## Testing for feature correlation

In [ ]:
# check correlation in training data
corr_sem_openrate = np.corrcoef(
    X_train_f['past_open_rate'], 
    X_train_f['semantic_match_score']
)[0, 1]

print(f"Correlation: past_open_rate × semantic_match_score = {corr_sem_openrate:.4f}")

# also check in test data
corr_test = np.corrcoef(
    X_test_f['past_open_rate'],
    X_test_f['semantic_match_score']
)[0, 1]
print(f"Correlation (test):  past_open_rate × semantic_match_score = {corr_test:.4f}")

# also check novelty score
corr_nov = np.corrcoef(
    X_train_f['past_open_rate'],
    X_train_f['novelty_score']
)[0, 1]
print(f"Correlation: past_open_rate × novelty_score = {corr_nov:.4f}")

# full correlation matrix of key features
key_features = ['past_open_rate', 'past_opens', 'emails_before', 
                 'semantic_match_score', 'novelty_score']
corr_matrix = X_train_f[key_features].corr().round(3)
print("\nCorrelation matrix:")
print(corr_matrix.to_string())

## Age distribution histogram

In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(7, 5))

df_user_clean['age_clean'].dropna().hist(
    bins=30, ax=ax1, color='steelblue', edgecolor='white', alpha=0.85
)
ax1.set_title('Age Distribution of Customers', fontsize=13, fontweight='bold')
ax1.set_xlabel('Age')
ax1.set_ylabel('Number of Customers')
ax1.axvline(df_user_clean['age_clean'].median(), color='red', linestyle='--', 
             linewidth=1.5, label=f"Median: {df_user_clean['age_clean'].median():.0f}")
ax1.legend()
ax1.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('age_analysis.png', dpi=150, bbox_inches='tight')
plt.show()